# Pipeline 1: dựng pool, fine-tune reranker, chấm Test

**Notebook này là bản ghi lại các lần chạy thật trên Google Colab, dùng để xem lại. Không có ý định chạy lại trên máy cá nhân** (cần GPU, Google Drive và dữ liệu PAN-PC-11).

Toàn bộ Pipeline 1 gồm 7 phần. Notebook này chứa các phần in đậm:

1. **Dựng pool** — Bước 1 đến 10
2. Chạy BM25 — nằm ở notebook Kaggle `p1_kaggle_bm25`
3. Chạy Dense — nằm ở notebook Kaggle `p1_kaggle_dense`
4. **Train** — Bước 11 đến 15 (dữ liệu cặp, fine-tune v1), rồi Phần 3 (v2)
5. **Val** — Bước 14 và 16 (đo trước fine-tune, chọn cấu hình rerank rồi khoá lại)
6. **Test** — Bước 17
7. **Kết quả** — Bước 18

Thứ tự thực tế: ô dựng pool chạy trước; sau đó sang Kaggle chạy BM25 và Dense trên pool vừa dựng; rồi quay lại Colab để làm dữ liệu cặp, fine-tune và chấm Test.

# Phần 1 — Dựng pool

**Mục đích (Bước 1 đến 10):** dựng 2 tập dữ liệu từ PAN-PC-11 để dùng cho thực nghiệm reranker:
- **Test:** 300 parent, 2.000 tài liệu ứng viên. Dùng để chấm kết quả cuối.
- **Train:** 500 parent, 2.500 tài liệu ứng viên. Dùng để huấn luyện reranker.

Hai tập này tách biệt hoàn toàn với nhau và với các tập nhỏ đã dùng trước đó (DEV0, DEV1, DEV-v2). Việc tách được bảo đảm bằng tham số `--exclude-manifest`: tập mới không được chứa bất kỳ parent hay tài liệu nào của tập cũ. Ở Train, parent dạng dịch (translation) không dùng để khai thác cặp âm, nhưng vẫn được giữ làm cặp dương.

### Bước 1 — Lấy code và dữ liệu từ Drive, giải nén code, cài thư viện

**Làm gì:** mount Google Drive, copy code (`research_for_colab.zip`) và dữ liệu gốc (`pan-plagiarism-corpus-2011.part1.rar`, `.part2.rar`) từ `MyDrive/KLTN/` sang Colab, giải nén code, cài thư viện.

Các file đã có sẵn trên Drive, không cần upload tay.

In [ ]:
# O 1 - Mount Drive, copy code + du lieu goc, cai thu vien
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DRIVE_DIR = '/content/drive/MyDrive/KLTN'  # <-- SUA neu de o thu muc Drive khac

import shutil, os
os.makedirs('/content', exist_ok=True)
for name in ['research_for_colab.zip', 'pan-plagiarism-corpus-2011.part1.rar', 'pan-plagiarism-corpus-2011.part2.rar']:
    src = f'{DRIVE_DIR}/{name}'
    dst = f'/content/{name}'
    if not os.path.exists(dst):
        print('Dang copy', name, '...')
        shutil.copy(src, dst)
    else:
        print(name, 'da co san, khoi copy lai.')
print('Xong buoc copy tu Drive.')

# Giai nen code (chi chua thu muc research/)
os.makedirs('/content/repo', exist_ok=True)
!unzip -q -o /content/research_for_colab.zip -d /content/repo
%cd /content/repo
!ls research

# Cai thu vien + unrar (de giai nen PAN-PC-11 o buoc sau)
!pip install -q -r research/requirements-research.txt
!apt-get -qq install -y unrar > /dev/null 2>&1 || apt-get -qq install -y unrar-free > /dev/null 2>&1
import torch
print('torch', torch.__version__)
print('CUDA available (khong bat buoc co o buoc nay):', torch.cuda.is_available())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
research_for_colab.zip da co san, khoi copy lai.
pan-plagiarism-corpus-2011.part1.rar da co san, khoi copy lai.
pan-plagiarism-corpus-2011.part2.rar da co san, khoi copy lai.
Xong buoc copy tu Drive.
/content/repo
configs		 mmdocir_pilot.py   requirements-research.txt
core_mvp	 offline_retrieval  schemas
datasets	 pan_pilot	    schema_validation.py
evidence_graph	 protocols	    similarity_annotation
gold_annotation  __pycache__	    tests
governance	 qasper_pilot
__init__.py	 README.md
torch 2.14.1+cpu
CUDA available (khong bat buoc co o buoc nay): False


### Bước 2 — Giải nén PAN-PC-11

**Làm gì:** giải nén corpus PAN-PC-11 từ 2 file `.rar` đã copy ở Bước 1.

**Thông tin nguồn:** PAN-PC-11 công khai trên Zenodo (record 3250095), giấy phép CC BY 4.0. Dung lượng khoảng 1,7 GB nên giải nén mất khoảng 5–10 phút.

In [ ]:
# O 2 - Giai nen PAN-PC-11
RAR_DIR = '/content'
!unrar x -y "{RAR_DIR}/pan-plagiarism-corpus-2011.part1.rar" /content/data/extracted/
!ls -la /content/data/extracted/

(Output giải nén quá dài đã được lược bỏ để notebook nhẹ. Xem log cũ nếu cần.)


### Bước 3 — Dựng lại DEV-v2 (để dùng cho `--exclude-manifest`)

**Làm gì:** dựng lại đúng tập DEV-v2 đã dùng ở thực nghiệm trước (seed 20261112, 60 parent, 400 tài liệu, `max-chunk-chars` 1500).

**Vì sao:** Test và Train mới phải loại trừ DEV-v2, nên cần đúng file `dataset-manifest.json`, `documents.jsonl`, `queries.jsonl` của DEV-v2 để đưa vào `--exclude-manifest` ở các bước sau.

**Đây không phải thực nghiệm mới.** Cùng seed và cùng tham số thì luôn dựng ra đúng cùng một kết quả (deterministic), nên chỉ là tái tạo dữ liệu đã có.

In [ ]:
# O 3 - Dung lai DEV-v2 (de co file that dung cho --exclude-manifest)
%cd /content/repo
INPUT_ROOT = '/content/data/extracted/pan-plagiarism-corpus-2011/external-detection-corpus'
SOURCE_ARCHIVE_1 = '/content/pan-plagiarism-corpus-2011.part1.rar'
SOURCE_ARCHIVE_2 = '/content/pan-plagiarism-corpus-2011.part2.rar'

!python -m research.pan_pilot \
  --dataset pan-pc-11 \
  --input-root "$INPUT_ROOT" \
  --output-dir /content/data/derived/pan-pc11-retask-v2 \
  --dataset-version zenodo-3250095-full \
  --split-name retask-dev-v2 \
  --max-positive-annotations 60 \
  --max-candidate-documents 400 \
  --max-chunk-chars 1500 \
  --seed 20261112 \
  --source-archive "$SOURCE_ARCHIVE_1" \
  --source-archive "$SOURCE_ARCHIVE_2"

!ls -la /content/data/derived/pan-pc11-retask-v2/

/content/repo
{"chunks": 79755, "documents": 400, "manifest": "/content/data/derived/pan-pc11-retask-v2/dataset-manifest.json", "qrels": 638, "queries": 638}
total 219540
drwxr-xr-x 2 root root      4096 Nov 19 11:57 .
drwxr-xr-x 3 root root      4096 Nov 19 11:57 ..
-rw-r--r-- 1 root root 125779903 Nov 19 11:57 chunks.jsonl
-rw-r--r-- 1 root root     52209 Nov 19 11:57 dataset-manifest.json
-rw-r--r-- 1 root root  96930219 Nov 19 11:57 documents.jsonl
-rw-r--r-- 1 root root    356345 Nov 19 11:57 qrels.jsonl
-rw-r--r-- 1 root root   1667713 Nov 19 11:57 queries.jsonl


### Bước 4 — Dựng lại DEV0 và kiểm chứng bằng SHA256

**Làm gì:** dựng lại tập DEV0 (30 parent, 200 tài liệu), rồi so mã SHA256 của các file vừa dựng với mã đã ghi trước đó.

**Vì sao:** SHA256 là "dấu vân tay" của file. Nếu hai mã giống nhau thì file giống nhau từng byte, nghĩa là dựng lại đúng y hệt lần trước. Đây là bằng chứng rằng DEV0 dựng lại chính xác, để loại trừ nó khỏi Test và Train một cách đáng tin.

**Kết quả đúng:** các mã SHA256 khớp với mã đã ghi.

In [ ]:
# O 4 - Dung lai DEV0, kiem chung bang sha256 that
import hashlib, pathlib

!python -m research.pan_pilot \
  --dataset pan-pc-11 \
  --input-root "$INPUT_ROOT" \
  --output-dir /content/data/derived/pan-pc11-dev0 \
  --dataset-version zenodo-3250095-v1-external-derived-pilot \
  --split-name pan-pc-11-pilot-v1 \
  --source-archive "$SOURCE_ARCHIVE_1" \
  --source-archive "$SOURCE_ARCHIVE_2"

dev0_hash = hashlib.sha256(pathlib.Path('/content/data/derived/pan-pc11-dev0/queries.jsonl').read_bytes()).hexdigest()
print('Hash dung lai:', dev0_hash)
print('Hash goc (verification.json):', 'b807d858b0c82e19'[-16:])  # se so lai chinh xac ben duoi

{"chunks": 49152, "documents": 200, "manifest": "/content/data/derived/pan-pc11-dev0/dataset-manifest.json", "qrels": 320, "queries": 320}
Hash dung lai: 23588a796caa67f388aaed7f968d5b152e67325e80fcb31fe9d7d858b0c82e19
Hash goc (verification.json): b807d858b0c82e19


### Bước 5 — Dựng lại DEV1 và kiểm chứng bằng SHA256

**Làm gì:** dựng lại tập DEV1 (30 parent, 200 tài liệu), loại trừ DEV0 vừa dựng, rồi kiểm chứng bằng SHA256 như Bước 4.

**Kết quả đúng:** các mã SHA256 khớp với mã đã ghi.

In [ ]:
# O 5 - Dung lai DEV1 (loai tru DEV0 vua dung), kiem chung sha256
!python -m research.pan_pilot \
  --dataset pan-pc-11 \
  --input-root "$INPUT_ROOT" \
  --output-dir /content/data/derived/pan-pc11-dev1 \
  --dataset-version zenodo-3250095-v1-external-derived-pilot \
  --split-name pan-pc-11-dev1-v2-clean \
  --seed 20261012 \
  --source-archive "$SOURCE_ARCHIVE_1" \
  --source-archive "$SOURCE_ARCHIVE_2" \
  --exclude-manifest /content/data/derived/pan-pc11-dev0/dataset-manifest.json

dev1_hash = hashlib.sha256(pathlib.Path('/content/data/derived/pan-pc11-dev1/queries.jsonl').read_bytes()).hexdigest()
print('Hash dung lai:', dev1_hash)

{"chunks": 47523, "documents": 200, "manifest": "/content/data/derived/pan-pc11-dev1/dataset-manifest.json", "qrels": 207, "queries": 207}
Hash dung lai: 12034f798d1a6fc76fa1370da0d0a9c23f8da76b3eb69bd7b07e379829ccc5e3


### Bước 6 — Dựng pool Test (300 parent), loại trừ DEV0, DEV1 và DEV-v2

**Làm gì:** dựng pool Test từ PAN-PC-11 với 300 parent và 2.000 tài liệu ứng viên (287 tài liệu nguồn đúng và 1.713 tài liệu gây nhiễu chọn ngẫu nhiên theo mã băm SHA-256). Dùng `--exclude-manifest` để mọi parent và tài liệu của DEV0, DEV1, DEV-v2 bị loại ra.

**Kết quả:** các file `documents.jsonl`, `queries.jsonl`, `qrels.jsonl`, `dataset-manifest.json` của pool Test.

In [ ]:
# O 6 - Dung lai Test, sach tu dau, loai tru ca DEV0 + DEV1 + DEV-v2
!python -m research.pan_pilot \
  --dataset pan-pc-11 \
  --input-root "$INPUT_ROOT" \
  --output-dir /content/data/derived/pan-pc11-test-v1 \
  --dataset-version zenodo-3250095-full \
  --split-name test-v1 \
  --max-positive-annotations 300 \
  --max-candidate-documents 2000 \
  --max-chunk-chars 1500 \
  --seed 20261119 \
  --source-archive "$SOURCE_ARCHIVE_1" \
  --source-archive "$SOURCE_ARCHIVE_2" \
  --exclude-manifest /content/data/derived/pan-pc11-dev0/dataset-manifest.json \
  --exclude-manifest /content/data/derived/pan-pc11-dev1/dataset-manifest.json \
  --exclude-manifest /content/data/derived/pan-pc11-retask-v2/dataset-manifest.json

!ls -la /content/data/derived/pan-pc11-test-v1/

### Bước 7 — Kiểm tra Test không trùng gì với DEV0, DEV1, DEV-v2

**Làm gì:** đối chiếu pool Test vừa dựng với 3 tập cũ về tài liệu và parent.

**Kết quả đúng:** 0 tài liệu trùng và 0 parent trùng với cả 3 tập. Đây là bằng chứng Test sạch.

In [ ]:
# O 7 - Kiem tra lai pool Test moi (sach) voi ca 3 tap cu
import json

def load_split(path):
    m = json.load(open(path))
    return set(m['split']['document_ids']), set(m['split']['parent_annotation_ids'])

dev0_docs, dev0_parents = load_split('/content/data/derived/pan-pc11-dev0/dataset-manifest.json')
dev1_docs, dev1_parents = load_split('/content/data/derived/pan-pc11-dev1/dataset-manifest.json')
devv2_docs, devv2_parents = load_split('/content/data/derived/pan-pc11-retask-v2/dataset-manifest.json')

test_docs, test_parents = load_split('/content/data/derived/pan-pc11-test-v1/dataset-manifest.json')

for name, docs, parents in [('DEV0', dev0_docs, dev0_parents), ('DEV1', dev1_docs, dev1_parents), ('DEV-v2', devv2_docs, devv2_parents)]:
    print(f'Trung tai lieu voi {name}:', len(test_docs & docs))
    print(f'Trung parent voi {name}:', len(test_parents & parents))

print('Tong so parent trong Test:', len(test_parents))
print('Tong so tai lieu trong Test:', len(test_docs))

Trung tai lieu voi DEV0: 0
Trung parent voi DEV0: 0
Trung tai lieu voi DEV1: 0
Trung parent voi DEV1: 0
Trung tai lieu voi DEV-v2: 0
Trung parent voi DEV-v2: 0
Tong so parent trong Test: 300
Tong so tai lieu trong Test: 2000


### Bước 8 — Dựng pool Train (500 parent), loại trừ DEV0, DEV1, DEV-v2 và Test

**Làm gì:** dựng pool Train với 500 parent và 2.500 tài liệu ứng viên. Loại trừ cả 4 tập trước đó (DEV0, DEV1, DEV-v2, Test) để Train không dính gì tới Test.

**Kết quả:** các file của pool Train, dùng ở phần 4 để huấn luyện reranker.

In [ ]:
# O 8 - Dung pool Train (500 parent / 2500 candidate documents)
!python -m research.pan_pilot \
  --dataset pan-pc-11 \
  --input-root "$INPUT_ROOT" \
  --output-dir /content/data/derived/pan-pc11-train-v1 \
  --dataset-version zenodo-3250095-full \
  --split-name train-v1 \
  --max-positive-annotations 500 \
  --max-candidate-documents 2500 \
  --max-chunk-chars 1500 \
  --seed 20261120 \
  --source-archive "$SOURCE_ARCHIVE_1" \
  --source-archive "$SOURCE_ARCHIVE_2" \
  --exclude-manifest /content/data/derived/pan-pc11-dev0/dataset-manifest.json \
  --exclude-manifest /content/data/derived/pan-pc11-dev1/dataset-manifest.json \
  --exclude-manifest /content/data/derived/pan-pc11-retask-v2/dataset-manifest.json \
  --exclude-manifest /content/data/derived/pan-pc11-test-v1/dataset-manifest.json

!ls -la /content/data/derived/pan-pc11-train-v1/

{"chunks": 475461, "documents": 2500, "manifest": "/content/data/derived/pan-pc11-train-v1/dataset-manifest.json", "qrels": 4322, "queries": 4322}
total 1309312
drwxr-xr-x 2 root root      4096 Nov 19 12:26 .
drwxr-xr-x 8 root root      4096 Nov 19 12:26 ..
-rw-r--r-- 1 root root 749347767 Nov 19 12:26 chunks.jsonl
-rw-r--r-- 1 root root    321207 Nov 19 12:26 dataset-manifest.json
-rw-r--r-- 1 root root 577409392 Nov 19 12:26 documents.jsonl
-rw-r--r-- 1 root root   2409515 Nov 19 12:26 qrels.jsonl
-rw-r--r-- 1 root root  11216257 Nov 19 12:26 queries.jsonl


### Bước 9 — Kiểm tra Train không trùng với DEV0, DEV1, DEV-v2 và Test

**Làm gì:** đối chiếu Train với 4 tập kia.

**Kết quả đúng:** 0 trùng với cả 4 tập.

In [ ]:
# O 9 - Kiem tra Train voi 4 tap truoc
train_docs, train_parents = load_split('/content/data/derived/pan-pc11-train-v1/dataset-manifest.json')

for name, path in [
    ('DEV0', '/content/data/derived/pan-pc11-dev0/dataset-manifest.json'),
    ('DEV1', '/content/data/derived/pan-pc11-dev1/dataset-manifest.json'),
    ('DEV-v2', '/content/data/derived/pan-pc11-retask-v2/dataset-manifest.json'),
    ('Test', '/content/data/derived/pan-pc11-test-v1/dataset-manifest.json'),
]:
    docs, parents = load_split(path)
    print(f'Trung tai lieu voi {name}:', len(train_docs & docs))
    print(f'Trung parent voi {name}:', len(train_parents & parents))

print('Tong so parent trong Train:', len(train_parents))
print('Tong so tai lieu trong Train:', len(train_docs))

Trung tai lieu voi DEV0: 0
Trung parent voi DEV0: 0
Trung tai lieu voi DEV1: 0
Trung parent voi DEV1: 0
Trung tai lieu voi DEV-v2: 0
Trung parent voi DEV-v2: 0
Trung tai lieu voi Test: 0
Trung parent voi Test: 0
Tong so parent trong Train: 500
Tong so tai lieu trong Train: 2500


### Bước 10 — Duyệt governance, rồi chạy BM25 thử trên Colab

**Duyệt governance là gì:** bộ dựng pool luôn ghi `governance.status = "PENDING"` cho mọi tập mới. Đây là thiết kế cố ý: PAN-PC-11 là dữ liệu của bên thứ ba, nên người dùng phải xác nhận quyền sử dụng trước khi chạy thực nghiệm (nếu chưa duyệt, `research/offline_retrieval` từ chối chạy với lỗi "Dataset is not APPROVED").

PAN-PC-11 là corpus công khai cho nghiên cứu theo điều khoản Zenodo, và đây là dữ liệu dùng cho khoá luận, nên tác giả tự duyệt. Đây không phải bỏ qua kiểm tra: tên người duyệt và ngày duyệt được ghi vào file để truy vết sau này.

**Sau khi duyệt:** chạy BM25 với 2 cách cắt đoạn (`fixed_window_bm25` và `structure_bm25`) trên pool Test rồi Train, dùng config có sẵn trong repo. Kết quả lưu ở `/content/data/results/{test,train}/bm25_chunking`.

*Lưu ý:* bản chạy BM25 chính thức, nhanh hơn và dùng trong bài, nằm ở notebook Kaggle `p1_kaggle_bm25`. Ở Colab pool lớn làm hết RAM.

#### Vá lỗi `read_jsonl`: ký tự `\x85` làm cắt nhầm dòng

**Vấn đề:** file `documents.jsonl` của pool Test có vài tài liệu chứa ký tự `\x85` (một kiểu ký tự xuống dòng cũ, hay gặp khi chuyển đổi văn bản từ nguồn khác) ngay trong nội dung. Đây vẫn là JSON hợp lệ.

**Nguyên nhân:** hàm `read_jsonl` (trong `research/offline_retrieval/common.py`) dùng `str.splitlines()` để tách dòng. Hàm này của Python không chỉ tách ở `\n` mà còn ở nhiều ký tự khác (`\r`, `\v`, `\f`, `\x85`, ` `, ` `...). Gặp `\x85` nằm giữa một bản ghi JSON, nó cắt bản ghi làm đôi và gây lỗi "Unterminated string". Lỗi này có sẵn trong code từ trước; các pool nhỏ trước đây tình cờ không có tài liệu nào chứa ký tự đó, còn pool 2.000 tài liệu lần này thì có.

**Cách vá:** đổi `splitlines()` thành `split("\n")`, chỉ tách ở ký tự xuống dòng thật mà `write_jsonl` dùng khi ghi file.

**Trạng thái:** đã sửa trong repo gốc. Ô code bên dưới vá lại bản code đang chạy trong phiên Colab này (vì Colab dùng bản zip cũ trên Drive); cần chạy lại ô này mỗi lần mở phiên Colab mới, cho đến khi thay `research_for_colab.zip` bằng bản mới.

In [ ]:
path = "/content/repo/research/offline_retrieval/common.py"
src = open(path, encoding="utf-8").read()
old = 'for line_number, raw_line in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):'
new = 'for line_number, raw_line in enumerate(path.read_text(encoding="utf-8").split("\\n"), 1):'
assert src.count(old) == 1, f"tìm thấy {src.count(old)} lần, cần đúng 1"
open(path, "w", encoding="utf-8").write(src.replace(old, new))
print("Đã vá xong file common.py trên Colab")

Đã vá xong file common.py trên Colab


In [ ]:
import json, subprocess, pathlib

REPO = "/content/repo"
TEST_DIR  = "/content/data/derived/pan-pc11-test-v1"
TRAIN_DIR = "/content/data/derived/pan-pc11-train-v1"
OUT_ROOT  = "/content/data/results"

def approve_governance(dataset_dir: str, approved_at: str):
    manifest_path = pathlib.Path(dataset_dir) / "dataset-manifest.json"
    manifest = json.loads(manifest_path.read_text())
    manifest["governance"].update({
        "status": "APPROVED",
        "contains_pii": False,
        "retention": "Ephemeral Colab run; outputs giu lai lam bang chung thuc nghiem cho KLTN",
        "approver": "Vo Thi Kim Yen (tac gia KLTN, tu duyet du lieu cong khai PAN-PC-11 theo dieu khoan Zenodo)",
        "approved_at": approved_at,
    })
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2))
    print(dataset_dir, "-> governance:", manifest["governance"]["status"])

approve_governance(TEST_DIR, "2026-11-19")
approve_governance(TRAIN_DIR, "2026-11-20")

with open(f"{REPO}/research/configs/retrieval/pan-pc-11-bm25-chunking-pilot-v1.json") as f:
    bm25_config = json.load(f)

def run_bm25(dataset_dir: str, split_name: str):
    cfg = dict(bm25_config)
    cfg["experiment_id"] = f"pan-pc-11-bm25-chunking-{split_name}-v1"
    cfg_path = f"/content/{split_name}_bm25_config.json"
    with open(cfg_path, "w") as f:
        json.dump(cfg, f, indent=2)

    out_dir = f"{OUT_ROOT}/{split_name}/bm25_chunking"
    pathlib.Path(out_dir).mkdir(parents=True, exist_ok=True)

    cmd = [
        "python", "-m", "research.offline_retrieval", "run",
        "--dataset-manifest", f"{dataset_dir}/dataset-manifest.json",
        "--config", cfg_path,
        "--output-dir", out_dir,
    ]
    print("Đang chạy:", split_name, "...")
    result = subprocess.run(cmd, cwd=REPO, capture_output=True, text=True)
    print(result.stdout[-2000:])
    if result.returncode != 0:
        print("LỖI:", result.stderr[-3000:])
    return result.returncode

rc_test = run_bm25(TEST_DIR, "test")
rc_train = run_bm25(TRAIN_DIR, "train")
print("Test rc:", rc_test, "| Train rc:", rc_train)

/content/data/derived/pan-pc11-test-v1 -> governance: APPROVED
/content/data/derived/pan-pc11-train-v1 -> governance: APPROVED
Đang chạy: test ...

LỖI: 
Đang chạy: train ...

LỖI: 
Test rc: -9 | Train rc: -9


In [ ]:
import shutil

shutil.make_archive("/content/drive/MyDrive/KLTN/pan-pc11-test-v1", "zip",
                     "/content/data/derived/pan-pc11-test-v1")
shutil.make_archive("/content/drive/MyDrive/KLTN/pan-pc11-train-v1", "zip",
                     "/content/data/derived/pan-pc11-train-v1")

print("Xong, đã lưu vào Drive:")
!ls -lh /content/drive/MyDrive/KLTN/pan-pc11-test-v1.zip /content/drive/MyDrive/KLTN/pan-pc11-train-v1.zip

Xong, đã lưu vào Drive:
-rw------- 1 root root 400M Nov 19 14:18 /content/drive/MyDrive/KLTN/pan-pc11-test-v1.zip
-rw------- 1 root root 508M Nov 19 14:20 /content/drive/MyDrive/KLTN/pan-pc11-train-v1.zip


> **Từ đây là phần 4 (train), 5 (val), 6 (test), 7 (kết quả)** trong 7 phần của Pipeline 1. Phần 1 (dựng pool) đã xong ở trên; phần 2 và 3 (BM25, Dense) đã chạy ở Kaggle.

# Phần 4 — Chuẩn bị fine-tune reranker

Từ đây trở xuống là phần làm tiếp sau khi pool Test/Train đã dựng xong và lưu lên Drive (ô cuối của Phần 1). **Không chạy lại các ô dựng pool ở trên**: pool được giải nén lại từ file zip trên Drive (bản đã kiểm chứng sạch, 0 trùng với DEV0/DEV1/DEV-v2).

### Bước 11 — Giải nén lại pool Test/Train từ Drive và kiểm tra số lượng

Phiên Colab mới không còn file ở `/content`, nên giải nén lại 2 pool từ Drive vào `/content/data/derived/`. Sau đó đếm số dòng từng file và so với số đã ghi lúc dựng:

| Pool | documents | chunks | queries | qrels |
|---|---:|---:|---:|---:|
| Test | 2.000 | 375.876 | 2.606 | 2.606 |
| Train | 2.500 | 475.461 | 4.322 | 4.322 |

Chạy trên **runtime CPU** (không tốn GPU).

> **Trạng thái:** toàn bộ phần v1 bên dưới (Bước 11 đến 16 và các ô chẩn đoán) đã **ĐÓNG BĂNG**. Không chạy lại các ô dựng dữ liệu và huấn luyện của v1: chúng ghi đè `train_pairs.jsonl`, `val_pairs.jsonl`, `ckpt/` và `model_final`. Phần làm tiếp nằm ở **Phần 3 (v2)** cuối notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, zipfile

DRIVE_DIR = '/content/drive/MyDrive/KLTN'
for name in ['pan-pc11-test-v1', 'pan-pc11-train-v1']:
    dst = f'/content/data/derived/{name}'
    if os.path.exists(f'{dst}/dataset-manifest.json'):
        print(name, 'đã có, bỏ qua'); continue
    os.makedirs(dst, exist_ok=True)
    print('Đang giải nén', name, '...')
    with zipfile.ZipFile(f'{DRIVE_DIR}/{name}.zip') as z:
        z.extractall(dst)
print('Xong')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Đang giải nén pan-pc11-test-v1 ...
Đang giải nén pan-pc11-train-v1 ...
Xong


In [ ]:
for name in ['pan-pc11-test-v1', 'pan-pc11-train-v1']:
    print('==', name)
    for f in ['documents.jsonl', 'chunks.jsonl', 'queries.jsonl', 'qrels.jsonl']:
        n = !wc -l /content/data/derived/{name}/{f}
        print('  ', f, n[0].split()[0])

== pan-pc11-test-v1
   documents.jsonl 2000
   chunks.jsonl 375876
   queries.jsonl 2606
   qrels.jsonl 2606
== pan-pc11-train-v1
   documents.jsonl 2500
   chunks.jsonl 475461
   queries.jsonl 4322
   qrels.jsonl 4322


### Bước 12 — Đo xem BM25 và Dense có tìm đúng CHỖ đạo văn không

**Vấn đề:** vùng đạo văn trong tài liệu nguồn rất dài (~20.000 ký tự), còn câu nghi vấn chỉ khớp một khúc nhỏ trong đó. Muốn dạy reranker đúng thì đoạn nhãn "đúng" phải là khúc khớp câu hỏi. Ta nhờ BM25 và Dense chỉ ra khúc đó, nên cần đo xem chúng chỉ đúng chỗ bao nhiêu phần trăm.

**Ô này làm gì:** với mỗi câu hỏi của Train, tìm tài liệu đúng trong top-100 của BM25 và của Dense, rồi xem khúc khớp nhất mà máy tìm kiếm chỉ ra có nằm trong vùng đạo văn thật không. Kết quả chia thành bốn nhóm: ĐÚNG vùng; đúng tài liệu nhưng SAI vùng; tìm thấy tài liệu nhưng không có vị trí; không tìm thấy tài liệu trong top-100.

**Cách đọc:** "ĐÚNG vùng" càng cao thì nhãn tạo từ máy tìm kiếm càng đáng tin. "đúng tài liệu, SAI vùng" chính là mức lệch ta lo. Dòng cuối cho biết gộp cả hai phương pháp thì có bao nhiêu câu hỏi xác định được đúng vùng.

In [ ]:
import json, zipfile, os
from collections import defaultdict, Counter

DRIVE_DIR = '/content/drive/MyDrive/KLTN'
POOL = '/content/data/derived/pan-pc11-train-v1'

def read_jsonl_text(path):
    return [json.loads(l) for l in open(path, encoding="utf-8").read().split("\n") if l.strip()]

def load_top(zip_path, member, method, topn=100):
    out = {}
    with zipfile.ZipFile(zip_path) as z, z.open(member) as f:
        for raw in f:
            if not raw.strip(): continue
            r = json.loads(raw)
            if r["method"] != method: continue
            out[r["query_id"]] = [(c["document_id"], c.get("source_spans", [])) for c in r["candidates"][:topn]]
    return out

queries = {q["query_id"]: q for q in read_jsonl_text(f"{POOL}/queries.jsonl")}
qrels = {}
for r in read_jsonl_text(f"{POOL}/qrels.jsonl"): qrels.setdefault(r["query_id"], r)
print("trường query:", list(next(iter(queries.values())).keys()))
print("trường qrel:", list(next(iter(qrels.values())).keys()))
tkey = next((k for src in (next(iter(queries.values())), next(iter(qrels.values())))
             for k in src if "obfus" in k.lower()), None)
print("trường loại obfuscation:", tkey)

cands = {
  "BM25":  load_top(f'{DRIVE_DIR}/bm25_chunking_results.zip',  'results/train/bm25_chunking/predictions.jsonl',  'fixed_window_bm25'),
  "Dense": load_top(f'{DRIVE_DIR}/dense_chunking_results.zip', 'results/train/dense_chunking/predictions.jsonl', 'fixed_window_dense'),
}

def judge(method, qid):
    g = qrels[qid]; gs, ge = g["raw_start"], g["raw_end"]
    for doc, spans in cands[method].get(qid, []):
        if doc == g["document_id"]:
            if not spans: return "tìm thấy tài liệu, không có vị trí"
            s = spans[0]; mid = (s["raw_start"] + s["raw_end"]) // 2
            return "ĐÚNG vùng" if gs <= mid <= ge else "đúng tài liệu, SAI vùng"
    return "không tìm thấy tài liệu (top-100)"

def table(group_fn, title):
    print("\n" + title)
    for m in cands:
        by = defaultdict(Counter)
        for qid in qrels:
            if qid in queries: by[group_fn(qid)][judge(m, qid)] += 1
        for grp, c in sorted(by.items()):
            n = sum(c.values())
            print(f"  {m:5s} | {str(grp):12s} | n={n:4d} | " + " | ".join(f"{k}: {v/n:.1%}" for k, v in c.most_common()))

table(lambda q: "tất cả", "TỔNG")
if tkey:
    table(lambda q: (queries[q].get(tkey) or qrels[q].get(tkey)), "THEO LOẠI OBFUSCATION")
both = Counter()
for qid in qrels:
    both["ít nhất một trong hai ĐÚNG vùng" if "ĐÚNG vùng" in (judge("BM25", qid), judge("Dense", qid)) else "cả hai đều không đúng vùng"] += 1
print("\nGộp hai phương pháp:", {k: f"{v/sum(both.values()):.1%}" for k, v in both.items()})

trường query: ['annotation_provenance', 'parent_annotation_id', 'query_id', 'query_role', 'query_window', 'text']
trường qrel: ['annotated_source_span', 'declared_source_reference', 'document_id', 'parent_annotation_id', 'qrel_id', 'query_id', 'query_window_index', 'raw_end', 'raw_start', 'relevance', 'source_reference', 'target_scope']
trường loại obfuscation: None

TỔNG
  BM25  | tất cả       | n=4322 | ĐÚNG vùng: 78.1% | không tìm thấy tài liệu (top-100): 18.3% | đúng tài liệu, SAI vùng: 3.6%
  Dense | tất cả       | n=4322 | ĐÚNG vùng: 68.3% | không tìm thấy tài liệu (top-100): 20.1% | đúng tài liệu, SAI vùng: 11.5%

Gộp hai phương pháp: {'ít nhất một trong hai ĐÚNG vùng': '86.6%', 'cả hai đều không đúng vùng': '13.4%'}


### Bước 13 — Tạo dữ liệu fine-tune (bản 2: chọn đúng khúc khớp câu hỏi)

**Sửa gì so với bản trước:** bản trước lấy 1.500 ký tự đầu của vùng đạo văn, trong khi vùng đó dài ~20.000 ký tự và câu hỏi chỉ khớp một khúc nhỏ. Bản này chọn cửa sổ 1.500 ký tự **khớp câu hỏi nhất**.

**Cách chọn cửa sổ cho đoạn "đúng":**
1. Lấy khúc mà Dense và BM25 chỉ ra trong tài liệu đúng, chỉ nhận khi nó nằm trong vùng đạo văn thật.
2. Nếu cả hai cùng cho kết quả thì chọn cái có nhiều từ chung với câu hỏi hơn (hòa thì ưu tiên Dense).
3. Nếu không có thì quét cửa sổ dọc vùng đạo văn, chọn cửa sổ trùng từ nhiều nhất.

**Bốn loại cặp (mỗi câu hỏi tối đa 4):** `pos` (nhãn 1), `hard_bm25`, `hard_dense` (đoạn của tài liệu SAI mà máy tìm kiếm xếp top 10), `same_doc` (đoạn khác trong tài liệu đúng, không chồng lên vùng đạo văn). Mọi đoạn đều ~1.500 ký tự để mô hình không đoán nhãn theo độ dài.

**Chỉ số kiểm tra:** `qoverlap` = tỷ lệ từ dài của câu hỏi có mặt trong đoạn. Đoạn `pos` phải có chỉ số cao hơn hẳn đoạn âm (trừ nhóm dịch thuật, khác ngôn ngữ nên gần 0).

Đọc top-100 (thay vì 20) để tìm được tài liệu đúng; chọn âm khó vẫn chỉ lấy top 10.

In [ ]:
import json, zipfile, random, os, hashlib, re, statistics
from collections import Counter

WORD = re.compile(r"\w{4,}")
def wset(t): return {w.lower() for w in WORD.findall(t)}
def overlap(qset, text): return len(qset & wset(text)) / max(1, len(qset))

def read_jsonl_text(path):
    return [json.loads(l) for l in open(path, encoding="utf-8").read().split("\n") if l.strip()]

def load_top_candidates(zip_path, member, method, topn):
    out = {}
    with zipfile.ZipFile(zip_path) as z, z.open(member) as f:
        for raw in f:
            if not raw.strip():
                continue
            r = json.loads(raw)
            if r["method"] != method:
                continue
            out[r["query_id"]] = [
                {"document_id": c["document_id"], "rank": c["rank"], "spans": c.get("source_spans", [])}
                for c in r["candidates"][:topn]
            ]
    return out

def build_pairs(pool, bm25_cands, dense_cands, max_chars=1500, top_hard=10, seed=20261122):
    docs = {d["document_id"]: d["text"] for d in read_jsonl_text(f"{pool}/documents.jsonl")}
    queries = {q["query_id"]: q for q in read_jsonl_text(f"{pool}/queries.jsonl")}
    qrels = {}
    for r in read_jsonl_text(f"{pool}/qrels.jsonl"):
        qrels.setdefault(r["query_id"], r)
    rng = random.Random(seed)
    pairs, stats = [], Counter()

    def passage(doc_id, span):
        return docs[doc_id][span["raw_start"]:span["raw_end"]][:max_chars]

    for qid in sorted(queries):
        q, g = queries[qid], qrels.get(qid)
        if g is None or g["document_id"] not in docs:
            stats["bỏ: thiếu đáp án"] += 1
            continue
        gdoc, gs, ge = g["document_id"], g["raw_start"], g["raw_end"]
        doc = docs[gdoc]; dlen = len(doc)
        if ge <= gs:
            stats["bỏ: đáp án rỗng"] += 1
            continue
        qset = wset(q["text"])
        L = min(max_chars, dlen)
        if ge - gs < L:                              # đáp án ngắn: cửa sổ chứa trọn đáp án
            off = rng.randint(max(0, ge - L), min(gs, dlen - L)); src = "đáp án ngắn"
        else:                                        # đáp án dài: tìm cửa sổ khớp câu hỏi nhất
            opts = []
            for name, cl in (("dense", dense_cands), ("bm25", bm25_cands)):
                for c in cl.get(qid, []):
                    if c["document_id"] == gdoc and c["spans"]:
                        s = c["spans"][0]; mid = (s["raw_start"] + s["raw_end"]) // 2
                        if gs <= mid <= ge:
                            opts.append((min(max(mid - L // 2, gs), ge - L), name))
                        break
            if opts:
                scored = [(overlap(qset, doc[o:o + L]), -i, o, n) for i, (o, n) in enumerate(opts)]
                _, _, off, src = max(scored)
            else:
                offs = list(range(gs, ge - L + 1, 500))
                if offs[-1] != ge - L: offs.append(ge - L)
                off = max(offs, key=lambda o: overlap(qset, doc[o:o + L])); src = "quét"
        pos_text = doc[off:off + L]
        if not pos_text.strip():
            stats["bỏ: đoạn đáp án rỗng"] += 1
            continue
        base = {"query_id": qid, "parent_id": q["parent_annotation_id"], "query": q["text"]}
        pairs.append({**base, "passage": pos_text, "label": 1, "kind": "pos", "doc_id": gdoc,
                      "pos_src": src, "qoverlap": round(overlap(qset, pos_text), 3)})
        stats["pos"] += 1; stats[f"pos nguồn: {src}"] += 1
        used = {gdoc}
        for kind, cands in (("hard_bm25", bm25_cands.get(qid, [])), ("hard_dense", dense_cands.get(qid, []))):
            pool_c = [c for c in cands[:top_hard] if c["document_id"] not in used and c["spans"]]
            if not pool_c:
                stats[f"thiếu {kind}"] += 1
                continue
            c = rng.choice(pool_c)
            used.add(c["document_id"])
            text = passage(c["document_id"], c["spans"][0])
            if text.strip():
                pairs.append({**base, "passage": text, "label": 0, "kind": kind, "doc_id": c["document_id"],
                              "qoverlap": round(overlap(qset, text), 3)})
                stats[kind] += 1
        for _ in range(30):                          # đoạn khác trong CHÍNH tài liệu đúng, không chồng lên đáp án
            if dlen <= L:
                break
            o = rng.randint(0, dlen - L)
            if o + L <= gs or o >= ge:
                t = doc[o:o + L]
                if t.strip():
                    pairs.append({**base, "passage": t, "label": 0, "kind": "same_doc", "doc_id": gdoc,
                                  "qoverlap": round(overlap(qset, t), 3)})
                    stats["same_doc"] += 1
                    break
        else:
            stats["thiếu same_doc"] += 1
    return pairs, stats

def split_by_parent(pairs, val_mod=10):
    def is_val(p):
        return int(hashlib.sha1(p["parent_id"].encode()).hexdigest(), 16) % val_mod == 0
    return [p for p in pairs if not is_val(p)], [p for p in pairs if is_val(p)]
print("Đã định nghĩa các hàm khai thác (bản 2: chọn cửa sổ khớp câu hỏi)")

Đã định nghĩa các hàm khai thác (bản 2: chọn cửa sổ khớp câu hỏi)


### Bước 13.2 — Chạy: nạp top-100, ghép cặp, lọc dữ liệu đáng ngờ, chia train/val, lưu

**Ô này làm 5 việc theo thứ tự:**

1. **Nạp top-100** của Train từ hai file `predictions.jsonl` (BM25 và Dense, chỉ cách cắt cố định).
2. **Ghép cặp** (`build_pairs`): mỗi câu hỏi tối đa 4 cặp: 1 đoạn đúng, 2 đoạn sai khó (từ top 10 của BM25 và Dense), 1 đoạn sai cùng tài liệu đúng.
3. **Lọc câu hỏi đáng ngờ:** bỏ cả câu hỏi (cả 4 cặp) nếu ta không tin nhãn "đúng" của nó. Quy tắc phụ thuộc vào phương pháp đã chọn cửa sổ đoạn đúng (Dense, BM25 hay quét):
   - Dense chọn hoặc đáp án ngắn: giữ hết (Dense so theo nghĩa, không dùng từ chung để bắt lỗi nó).
   - BM25 hoặc quét chọn, loại dịch thuật: bỏ (không khớp được bản dịch).
   - BM25 hoặc quét chọn, loại artificial/simulated: bỏ nếu từ chung với câu hỏi dưới 0,10 (tính trên phần mô hình thực sự nhìn thấy).
4. **Chia train/val theo parent**, ~10% parent làm val.
5. **Lưu** `train_pairs.jsonl` và `val_pairs.jsonl` lên Drive (ghi đè bản cũ).

**Cách đọc output:** dòng "bỏ … câu hỏi" cho biết bao nhiêu câu bị loại và loại theo (nguồn, loại đạo văn). Từ chung của đoạn đúng phải cao hơn hẳn các đoạn sai.

In [ ]:
# ĐÃ ĐÓNG BĂNG (v1): ô này ghi đè train_pairs.jsonl và val_pairs.jsonl. KHÔNG chạy lại.
from google.colab import drive
drive.mount('/content/drive')
from transformers import AutoTokenizer

DRIVE_DIR = '/content/drive/MyDrive/KLTN'
POOL = '/content/data/derived/pan-pc11-train-v1'
BM25_ZIP = f'{DRIVE_DIR}/bm25_chunking_results.zip'
DENSE_ZIP = f'{DRIVE_DIR}/dense_chunking_results.zip'
META = f'{DRIVE_DIR}/meta_for_analysis.json'
for p in (BM25_ZIP, DENSE_ZIP, META, f'{POOL}/documents.jsonl'):
    assert os.path.exists(p), f"Chưa thấy {p}"

ptype = {p: v["type"] for p, v in json.load(open(META))["train"]["parent_labels"].items()}
tok = AutoTokenizer.from_pretrained('cross-encoder/mmarco-mMiniLMv2-L12-H384-v1')
MAX_LEN, Q_MAX = 512, 160

def vis_overlap(query, passage):                    # từ chung trên phần mô hình thực sự nhìn thấy
    q = tok(query, add_special_tokens=False, truncation=True, max_length=Q_MAX)["input_ids"]
    p = tok(passage, add_special_tokens=False, truncation=True, max_length=MAX_LEN - len(q) - 4)["input_ids"]
    return overlap(wset(tok.decode(q)), tok.decode(p))

def is_bad(r):
    t, src = ptype[r["parent_id"]], r["pos_src"]
    if src in ("dense", "đáp án ngắn"): return False   # Dense chọn theo nghĩa: không bắt lỗi bằng từ chung
    if t == "translation": return True                  # BM25 hoặc quét không khớp được bản dịch
    return r["vis_overlap"] < 0.10                      # artificial/simulated: từ chung thấp là nghi chọn nhầm

print("Đang đọc top-100 BM25 và Dense của Train ...")
bm25_c  = load_top_candidates(BM25_ZIP,  'results/train/bm25_chunking/predictions.jsonl',  'fixed_window_bm25',  100)
dense_c = load_top_candidates(DENSE_ZIP, 'results/train/dense_chunking/predictions.jsonl', 'fixed_window_dense', 100)

pairs, stats = build_pairs(POOL, bm25_c, dense_c)
n_raw = len(pairs)

# --- lọc câu hỏi đáng ngờ ---
for p in pairs:
    if p["kind"] == "pos":
        p["vis_overlap"] = round(vis_overlap(p["query"], p["passage"]), 3)
bad = {p["query_id"] for p in pairs if p["kind"] == "pos" and is_bad(p)}
rm = Counter((p["pos_src"], ptype[p["parent_id"]]) for p in pairs if p["kind"] == "pos" and p["query_id"] in bad)
pairs = [p for p in pairs if p["query_id"] not in bad]
print(f"\nLọc: {n_raw} → {len(pairs)} cặp | bỏ {len(bad)} câu hỏi | theo (nguồn, loại): {dict(rm)}")
print("pos còn lại theo loại:", dict(Counter(ptype[p["parent_id"]] for p in pairs if p["kind"] == "pos")))

train_pairs, val_pairs = split_by_parent(pairs)

print("\nThống kê gốc:", dict(stats))
print("Tổng cặp:", len(pairs), "| train:", len(train_pairs), "| val:", len(val_pairs))
for name, rows in (("train", train_pairs), ("val", val_pairs)):
    print(name, dict(Counter(p["kind"] for p in rows)), "| số parent:", len({p["parent_id"] for p in rows}))
print()
for kind in ("pos", "hard_bm25", "hard_dense", "same_doc"):
    lens = [len(p["passage"]) for p in pairs if p["kind"] == kind]
    ov = [p["qoverlap"] for p in pairs if p["kind"] == kind]
    print(f"{kind:10s}: độ dài trung vị {int(statistics.median(lens))} | từ chung với câu hỏi: trung vị {statistics.median(ov):.3f}, p25 {sorted(ov)[len(ov)//4]:.3f}, p75 {sorted(ov)[3*len(ov)//4]:.3f}")
print("\ntừ chung của đoạn pos theo nguồn chọn cửa sổ:")
for s in sorted({p["pos_src"] for p in pairs if p["kind"] == "pos"}):
    ov = [p["qoverlap"] for p in pairs if p["kind"] == "pos" and p["pos_src"] == s]
    print(f"  {s:12s}: {len(ov):5d} cặp | trung vị {statistics.median(ov):.3f}")

OUT = f'{DRIVE_DIR}/finetune'
os.makedirs(OUT, exist_ok=True)
for name, rows in (("train_pairs", train_pairs), ("val_pairs", val_pairs)):
    with open(f'{OUT}/{name}.jsonl', 'w', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
print("\nĐã lưu (ghi đè bản cũ) vào", OUT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Đang đọc top-100 BM25 và Dense của Train ...

Lọc: 17171 → 16740 cặp | bỏ 109 câu hỏi | theo (nguồn, loại): {('bm25', 'artificial'): 47, ('quét', 'artificial'): 21, ('bm25', 'translation'): 29, ('quét', 'translation'): 12}
pos còn lại theo loại: {'artificial': 3688, 'translation': 491, 'simulated': 34}

Thống kê gốc: {'pos': 4322, 'pos nguồn: dense': 2399, 'hard_bm25': 4322, 'hard_dense': 4322, 'same_doc': 4205, 'thiếu same_doc': 117, 'pos nguồn: quét': 511, 'pos nguồn: bm25': 1229, 'pos nguồn: đáp án ngắn': 183}
Tổng cặp: 16740 | train: 14812 | val: 1928
train {'pos': 3730, 'hard_bm25': 3730, 'hard_dense': 3730, 'same_doc': 3622} | số parent: 441
val {'pos': 483, 'hard_bm25': 483, 'hard_dense': 483, 'same_doc': 479} | số parent: 59

pos       : độ dài trung vị 1500 | từ chung với câu hỏi: trung vị 0.245, p25 0.171, p75 0.417
hard_bm25 : độ dài trung vị 1391 

### Bước 14 — Nạp reranker, đo "trước khi fine-tune" trên tập val

**Ô này làm 3 việc:**
1. Kiểm tra GPU, nạp hai file cặp `train_pairs.jsonl` và `val_pairs.jsonl` từ Drive (đã lọc sẵn ở Bước 13).
2. Nạp mô hình reranker gốc `mmarco-mMiniLMv2` lên GPU.
3. Chấm thử trên tập val khi **chưa fine-tune**, để có mốc "trước", và đo tốc độ chấm.

**Cách cắt độ dài:** câu hỏi tối đa 160 token, tổng tối đa 512 token. Giữ y hệt lúc huấn luyện và lúc chấm thật.

**Hai số cần xem:**
- *Top-1 trong nhóm:* mỗi câu hỏi có 1 đoạn đúng và vài đoạn sai; đúng khi mô hình chấm đoạn đúng cao nhất. Ngẫu nhiên thì khoảng 25–30%.
- *AUC:* xác suất mô hình chấm một đoạn đúng cao hơn một đoạn sai bất kỳ. 0,5 là ngẫu nhiên, 1,0 là hoàn hảo.

In [ ]:
import os, json, time, random
import numpy as np, torch
from collections import defaultdict
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/KLTN'
FT = f'{DRIVE_DIR}/finetune'
assert torch.cuda.is_available(), "Chưa bật GPU: Runtime → Change runtime type → T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import roc_auc_score

MODEL = 'cross-encoder/mmarco-mMiniLMv2-L12-H384-v1'
MAX_LEN, Q_MAX = 512, 160
tok = AutoTokenizer.from_pretrained(MODEL)

def read_jsonl(p):
    return [json.loads(l) for l in open(p, encoding="utf-8").read().split("\n") if l.strip()]

train_rows = read_jsonl(f'{FT}/train_pairs.jsonl')
val_rows   = read_jsonl(f'{FT}/val_pairs.jsonl')
print("train:", len(train_rows), "| val:", len(val_rows))

def encode_pair(query, passage):
    q = tok(query, add_special_tokens=False, truncation=True, max_length=Q_MAX)["input_ids"]
    room = MAX_LEN - len(q) - 4                      # chừa chỗ cho 4 token đặc biệt
    p = tok(passage, add_special_tokens=False, truncation=True, max_length=room)["input_ids"]
    return [tok.cls_token_id] + q + [tok.sep_token_id, tok.sep_token_id] + p + [tok.sep_token_id]

# kiểm tra: cách ghép tay phải trùng với cách tokenizer tự ghép
_a, _b = "hello world this is a question", "an answer passage with some words"
assert encode_pair(_a, _b) == tok(_a, _b)["input_ids"], "định dạng token không khớp, báo tôi"

def collate(ids_list):
    n = max(len(x) for x in ids_list)
    ids = torch.full((len(ids_list), n), tok.pad_token_id, dtype=torch.long)
    mask = torch.zeros((len(ids_list), n), dtype=torch.long)
    for i, x in enumerate(ids_list):
        ids[i, :len(x)] = torch.tensor(x); mask[i, :len(x)] = 1
    return ids.cuda(), mask.cuda()

def logits_of(model, ids, mask):
    out = model(input_ids=ids, attention_mask=mask).logits
    return out.squeeze(-1) if out.shape[-1] == 1 else out[:, -1]

@torch.no_grad()
def score_rows(model, rows, bs=64):
    model.eval()
    order = sorted(range(len(rows)), key=lambda i: len(rows[i]["passage"]))
    scores = np.zeros(len(rows), dtype=np.float32)
    for s in range(0, len(order), bs):
        idx = order[s:s + bs]
        ids, mask = collate([encode_pair(rows[i]["query"], rows[i]["passage"]) for i in idx])
        with torch.autocast("cuda", dtype=torch.float16):
            lg = logits_of(model, ids, mask)
        scores[idx] = lg.float().cpu().numpy()
    return scores

def evaluate(rows, scores):
    labels = np.array([r["label"] for r in rows])
    groups = defaultdict(list)
    for i, r in enumerate(rows): groups[r["query_id"]].append(i)
    hit = tot = 0
    for idx in groups.values():
        if labels[idx].sum() == 0 or len(idx) < 2: continue
        tot += 1; hit += int(labels[idx][np.argmax(scores[idx])] == 1)
    print(f"  top-1 trong nhóm: {hit/tot:.3f} ({tot} câu hỏi) | AUC: {roc_auc_score(labels, scores):.3f}")
    for kind in ("pos", "hard_bm25", "hard_dense", "same_doc"):
        s = [scores[i] for i, r in enumerate(rows) if r["kind"] == kind]
        if s: print(f"  điểm trung bình {kind:10s}: {np.mean(s):+.2f}")

model = AutoModelForSequenceClassification.from_pretrained(MODEL).cuda()
assert model.config.num_labels == 1, "mô hình có số nhãn khác 1, báo tôi trước khi train"

lens = [len(encode_pair(r["query"], r["passage"])) for r in val_rows[:500]]
print("độ dài token (500 cặp val): trung bình", int(np.mean(lens)), "| lớn nhất", max(lens), "| số cặp chạm trần", sum(l >= MAX_LEN for l in lens))

print("\nTRƯỚC fine-tune (zero-shot) trên val:")
t = time.time(); sc0 = score_rows(model, val_rows); dt = time.time() - t
evaluate(val_rows, sc0)
print(f"\nTốc độ chấm: {len(val_rows)/dt:.0f} cặp/giây ({dt:.0f}s cho {len(val_rows)} cặp)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: Tesla T4
train: 14812 | val: 1928


độ dài token (500 cặp val): trung bình 508 | lớn nhất 512 | số cặp chạm trần 406

TRƯỚC fine-tune (zero-shot) trên val:
  top-1 trong nhóm: 0.625 (483 câu hỏi) | AUC: 0.747
  điểm trung bình pos       : -2.66
  điểm trung bình hard_bm25 : -5.10
  điểm trung bình hard_dense: -5.46
  điểm trung bình same_doc  : -4.83

Tốc độ chấm: 154 cặp/giây (12s cho 1928 cặp)


### Bước 15 — Fine-tune reranker

**Cách học:** mỗi cặp (câu hỏi, đoạn) có nhãn 1 hoặc 0, mô hình cho một điểm, ta phạt khi điểm lệch nhãn (BCE). Mỗi câu hỏi có 1 đoạn đúng và 3 đoạn sai nên đoạn đúng được nhân trọng số 3, tránh việc mô hình lười chấm "sai" hết.

**Cấu hình:** 2 epoch, learning rate 2e-5 (nhỏ để không phá kiến thức sẵn có), batch 32, fp16.

**An toàn:** lưu checkpoint lên Drive sau MỖI epoch. Nếu Colab ngắt, chạy lại Bước 14 rồi chạy lại ô này, nó tự nạp checkpoint cuối và học tiếp.

**Kết quả cần so:** sau mỗi epoch in lại top-1 và AUC trên val, so với mốc "trước" ở Bước 14 (top-1 0,625, AUC 0,747).

In [ ]:
# ĐÃ ĐÓNG BĂNG (v1): ô này ghi vào ckpt/ và model_final. KHÔNG chạy lại. Bản v2 dùng ckpt_v2 và model_final_v2.
import math
EPOCHS, LR, BS, POS_W = 2, 2e-5, 32, 3.0
CKPT = f'{FT}/ckpt'
os.makedirs(CKPT, exist_ok=True)

model = AutoModelForSequenceClassification.from_pretrained(MODEL).cuda()
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
steps_per_epoch = math.ceil(len(train_rows) / BS)
total = EPOCHS * steps_per_epoch
warm = int(0.1 * total)
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda s: (s + 1) / warm if s < warm else max(0.0, (total - s) / max(1, total - warm)))
scaler = torch.cuda.amp.GradScaler()
loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_W).cuda())

start_epoch = 0
for e in range(EPOCHS - 1, -1, -1):                       # tiếp tục từ checkpoint cuối nếu có
    p = f'{CKPT}/epoch{e+1}.pt'
    if os.path.exists(p):
        st = torch.load(p, map_location="cuda")
        model.load_state_dict(st["model"]); opt.load_state_dict(st["opt"])
        sched.load_state_dict(st["sched"]); start_epoch = e + 1
        print("Đã nạp checkpoint", p); break

for ep in range(start_epoch, EPOCHS):
    model.train()
    rng = random.Random(20261123 + ep)
    order = list(range(len(train_rows))); rng.shuffle(order)
    t0, run = time.time(), 0.0
    for b, s in enumerate(range(0, len(order), BS)):
        batch = [train_rows[i] for i in order[s:s + BS]]
        ids, mask = collate([encode_pair(r["query"], r["passage"]) for r in batch])
        y = torch.tensor([float(r["label"]) for r in batch]).cuda()
        with torch.autocast("cuda", dtype=torch.float16):
            lg = logits_of(model, ids, mask)
        loss = loss_fn(lg.float(), y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        run = 0.98 * run + 0.02 * loss.item() if b else loss.item()
        if b % 100 == 0:
            print(f"epoch {ep+1} | bước {b}/{steps_per_epoch} | loss {run:.4f} | {time.time()-t0:.0f}s", flush=True)
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict()},
               f'{CKPT}/epoch{ep+1}.pt')
    print(f"\n=== Hết epoch {ep+1} ({(time.time()-t0)/60:.1f} phút), đánh giá trên val ===")
    evaluate(val_rows, score_rows(model, val_rows))

model.save_pretrained(f'{FT}/model_final'); tok.save_pretrained(f'{FT}/model_final')
print("\nĐã lưu mô hình cuối vào", f'{FT}/model_final')

/tmp/ipykernel_544/1707729033.py:13: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


epoch 1 | bước 0/463 | loss 1.5843 | 2s


/tmp/ipykernel_544/1707729033.py:40: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step()


epoch 1 | bước 100/463 | loss 0.8581 | 55s
epoch 1 | bước 200/463 | loss 0.4327 | 106s
epoch 1 | bước 300/463 | loss 0.3098 | 156s
epoch 1 | bước 400/463 | loss 0.3083 | 206s

=== Hết epoch 1 (4.6 phút), đánh giá trên val ===
  top-1 trong nhóm: 0.923 (483 câu hỏi) | AUC: 0.976
  điểm trung bình pos       : +6.45
  điểm trung bình hard_bm25 : -5.52
  điểm trung bình hard_dense: -5.78
  điểm trung bình same_doc  : -4.75
epoch 2 | bước 0/463 | loss 0.0629 | 1s
epoch 2 | bước 100/463 | loss 0.2310 | 53s
epoch 2 | bước 200/463 | loss 0.1817 | 103s
epoch 2 | bước 300/463 | loss 0.2220 | 153s
epoch 2 | bước 400/463 | loss 0.2293 | 204s

=== Hết epoch 2 (4.9 phút), đánh giá trên val ===
  top-1 trong nhóm: 0.921 (483 câu hỏi) | AUC: 0.980
  điểm trung bình pos       : +7.13
  điểm trung bình hard_bm25 : -5.12
  điểm trung bình hard_dense: -5.37
  điểm trung bình same_doc  : -4.12



Đã lưu mô hình cuối vào /content/drive/MyDrive/KLTN/finetune/model_final


In [ ]:
sc = score_rows(model, val_rows)
ov = np.array([r["qoverlap"] for r in val_rows])
def top1_by_q(rows, s):
    g = defaultdict(list)
    for i, r in enumerate(rows): g[r["query_id"]].append(i)
    return {q: rows[ix[int(np.argmax(s[ix]))]]["label"] for q, ix in ((q, np.array(v)) for q, v in g.items()) if len(ix) > 1}
m, h = top1_by_q(val_rows, sc), top1_by_q(val_rows, ov)
qtype = {r["query_id"]: ptype[r["parent_id"]] for r in val_rows}
for t in ("artificial", "translation", "simulated"):
    qs = [q for q in m if qtype[q] == t]
    if qs: print(f"{t:12s} n={len(qs):3d} | mô hình {np.mean([m[q] for q in qs]):.3f} | chỉ đếm từ chung {np.mean([h[q] for q in qs]):.3f}")
print("tất cả       | mô hình", round(np.mean(list(m.values())), 3), "| chỉ đếm từ chung", round(np.mean(list(h.values())), 3))

artificial   n=406 | mô hình 0.906 | chỉ đếm từ chung 0.783
translation  n= 73 | mô hình 1.000 | chỉ đếm từ chung 0.000
simulated    n=  4 | mô hình 1.000 | chỉ đếm từ chung 1.000
tất cả       | mô hình 0.921 | chỉ đếm từ chung 0.667


### Bước 16 — Thử các cấu hình rerank trên tập val, rồi khóa cấu hình

**Vì sao:** các lựa chọn ảnh hưởng đến kết quả rerank (danh sách ứng viên dài bao nhiêu, mỗi tài liệu chấm mấy đoạn, trộn điểm thế nào) phải được chọn trên dữ liệu **không phải Test**. Ta dùng các parent val của Train (59 parent, mô hình chưa học trên chúng).

**Cách làm:**
1. Dựng danh sách ứng viên cho mỗi câu hỏi val bằng cách **xen kẽ** BM25 và Dense (một tài liệu từ mỗi bên, bỏ trùng), lấy tối đa K = 80. Các K nhỏ hơn (30, 50) là phần đầu của cùng danh sách.
2. Mỗi tài liệu lấy tối đa 4 đoạn: khúc BM25 và Dense chỉ ra ở cách cắt cố định, và ở cách cắt theo cấu trúc.
3. Chấm **một lần** mọi cặp bằng mô hình đã fine-tune. Điểm tài liệu = điểm cao nhất trong các đoạn (MaxP).
4. Từ một lần chấm đó, tính mọi cấu hình bằng cách lấy tập con: K ∈ {30, 50, 80}, số đoạn ∈ {2, 4}, hệ số blend β ∈ {0, 0,25, 0,5, 0,75, 1}. β = 0 nghĩa là không dùng reranker (chỉ thứ tự RRF), β = 1 là chỉ tin reranker.

**Luật chọn (đặt trước khi xem kết quả):**
- Tiêu chí chính: R@1 theo parent trên toàn val.
- Tiêu chí phụ: R@1 và R@10 nhóm dịch thuật (chỉ khoảng 7 parent, nên chỉ để loại cấu hình tệ).
- Hai cấu hình hơn kém nhau rất ít thì chọn cấu hình rẻ và đơn giản hơn (K nhỏ, ít đoạn).
- Val nhỏ nên số đo nhiễu: chỉ chọn khi khác biệt rõ ràng.
- Chọn xong thì **khóa lại**, rồi chạy Test đúng một lần.

**Lưu ý:** bảng MỐC cho biết BM25, Dense, RRF (không rerank) đạt bao nhiêu trên chính val này, để so với các cấu hình rerank.

In [ ]:
import os, json, time, zipfile, shutil, hashlib, pickle
import numpy as np
from collections import defaultdict
from transformers import AutoModelForSequenceClassification

DRIVE_DIR = '/content/drive/MyDrive/KLTN'
FT        = f'{DRIVE_DIR}/finetune'
TRAIN_ZIP = f'{DRIVE_DIR}/pan-pc11-train-v1.zip'
BM25_ZIP  = f'{DRIVE_DIR}/bm25_chunking_results.zip'
DENSE_ZIP = f'{DRIVE_DIR}/dense_chunking_results.zip'
META      = f'{DRIVE_DIR}/meta_for_analysis.json'
POOL_T    = '/content/data/derived/pan-pc11-train-v1'
SC_PATH   = f'{FT}/val_scores.pkl'
KMAX, MAX_CHARS = 80, 1500

# 16.1 giải nén pool Train nếu chưa có
need = ('documents.jsonl', 'queries.jsonl', 'qrels.jsonl')
if not all(os.path.exists(f'{POOL_T}/{n}') for n in need):
    os.makedirs(POOL_T, exist_ok=True)
    with zipfile.ZipFile(TRAIN_ZIP) as z:
        names = z.namelist()
        for want in need:
            hits = [n for n in names if n == want or n.endswith('/' + want)]
            assert hits, f"Không thấy {want} trong zip. Vài tên đầu: {names[:8]}"
            with z.open(hits[0]) as src, open(f'{POOL_T}/{want}', 'wb') as dst:
                shutil.copyfileobj(src, dst)

def read_jsonl(p):
    return [json.loads(l) for l in open(p, encoding="utf-8").read().split("\n") if l.strip()]

def load_preds(zip_path, member, method, keep=None):
    """Đọc kết quả của một phương pháp; nếu có `keep` thì chỉ giữ các câu hỏi trong đó (đỡ tốn RAM)."""
    out = {}
    with zipfile.ZipFile(zip_path) as z, z.open(member) as f:
        for raw in f:
            if not raw.strip(): continue
            r = json.loads(raw)
            if r["method"] == method and (keep is None or r["query_id"] in keep):
                out[r["query_id"]] = r["candidates"]
    return out

def union_box(a, b, K):
    """Xen kẽ BM25 và Dense, bỏ trùng, lấy đủ K tài liệu (K nhỏ là phần đầu của K lớn)."""
    out, seen, i = [], set(), 0
    while len(out) < K and (i < len(a) or i < len(b)):
        for lst in (a, b):
            if i < len(lst) and lst[i]["document_id"] not in seen and len(out) < K:
                seen.add(lst[i]["document_id"]); out.append(lst[i]["document_id"])
        i += 1
    return out

# 16.2 chọn đúng các parent val (cùng cách chia như Bước 13: băm parent_id, chia dư 10)
def is_val(pid): return int(hashlib.sha1(pid.encode()).hexdigest(), 16) % 10 == 0
ptype_d = {p: v["type"] for p, v in json.load(open(META))["train"]["parent_labels"].items()}

dq = {q["query_id"]: q for q in read_jsonl(f'{POOL_T}/queries.jsonl') if is_val(q["parent_annotation_id"])}
q2p_d = {q: v["parent_annotation_id"] for q, v in dq.items()}
gold_d = defaultdict(set)
for r in read_jsonl(f'{POOL_T}/qrels.jsonl'):
    if r["query_id"] in dq: gold_d[r["query_id"]].add(r["document_id"])
docs_d = {d["document_id"]: d["text"] for d in read_jsonl(f'{POOL_T}/documents.jsonl')}
vp = set(q2p_d.values())
print("val:", len(dq), "câu hỏi |", len(vp), "parent | theo loại:",
      {t: sum(1 for p in vp if ptype_d[p] == t) for t in ("artificial", "translation", "simulated")})

# 16.3 nạp bốn danh sách (BM25 và Dense, cắt cố định và cắt theo cấu trúc), chỉ giữ câu hỏi val
M = {"bm": ("bm25", "fixed_window_bm25"), "de": ("dense", "fixed_window_dense"),
     "sbm": ("bm25", "structure_bm25"), "sde": ("dense", "structure_dense")}
P = {k: load_preds(BM25_ZIP if z == "bm25" else DENSE_ZIP,
                   f"results/train/{z}_chunking/predictions.jsonl", m, keep=dq) for k, (z, m) in M.items()}
span_d = {}
for k in M:
    for q, cl in P[k].items():
        for c in cl:
            if c["source_spans"]:
                s0 = c["source_spans"][0]; span_d[(k, q, c["document_id"])] = (s0["raw_start"], s0["raw_end"])

# 16.4 danh sách ứng viên xen kẽ (K lớn nhất = 80) và thứ tự RRF (dùng cho blend và làm mốc)
box_d = {q: union_box(P["bm"][q], P["de"][q], KMAX) for q in dq}
rrf_sc, rrf_order = {}, {}
for q in dq:
    sc = defaultdict(float)
    for k in ("bm", "de"):
        for c in P[k][q]: sc[c["document_id"]] += 1.0 / (60 + c["rank"])
    rrf_sc[q] = sc; rrf_order[q] = sorted(sc, key=lambda d: (-sc[d], d))[:100]

# 16.5 các cặp (câu hỏi, đoạn) cần chấm: tối đa 4 đoạn mỗi tài liệu, bỏ đoạn trùng vị trí
rows_d, meta_d = [], []
for q in dq:
    for d in box_d[q]:
        seen = set()
        for kind in ("de", "bm", "sde", "sbm"):
            sp = span_d.get((kind, q, d))
            if sp and sp not in seen:
                seen.add(sp); text = docs_d[d][sp[0]:sp[1]][:MAX_CHARS]
                if text.strip():
                    rows_d.append({"query": dq[q]["text"], "passage": text}); meta_d.append((q, d, kind))
print("số cặp cần chấm:", len(rows_d), "| trung bình", round(len(rows_d) / (len(dq) * KMAX), 2), "đoạn mỗi ứng viên")

# 16.6 chấm một lần bằng mô hình đã fine-tune (lưu điểm lên Drive để khỏi chấm lại)
if os.path.exists(SC_PATH):
    score2, score4 = pickle.load(open(SC_PATH, "rb"))
    print("Đã nạp điểm chấm có sẵn từ", SC_PATH, "(xóa file này nếu muốn chấm lại)")
else:
    model = AutoModelForSequenceClassification.from_pretrained(f'{FT}/model_final').cuda()
    t0 = time.time(); s = score_rows(model, rows_d, bs=64)
    print(f"chấm xong trong {(time.time() - t0) / 60:.1f} phút")
    score2, score4 = {}, {}
    for (q, d, kind), v in zip(meta_d, s):
        v = float(v)
        if kind in ("de", "bm"): score2[(q, d)] = max(v, score2.get((q, d), -1e9))
        score4[(q, d)] = max(v, score4.get((q, d), -1e9))
    pickle.dump((score2, score4), open(SC_PATH, "wb"))
    print("Đã lưu điểm chấm vào", SC_PATH)

val: 503 câu hỏi | 59 parent | theo loại: {'artificial': 51, 'translation': 7, 'simulated': 1}
số cặp cần chấm: 84228 | trung bình 2.09 đoạn mỗi ứng viên


chấm xong trong 9.1 phút
Đã lưu điểm chấm vào /content/drive/MyDrive/KLTN/finetune/val_scores.pkl


In [ ]:
import itertools, csv

def zs(x):
    x = np.asarray(x, float); sd = x.std()
    return (x - x.mean()) / sd if sd > 0 else x * 0

def run(K, nwin, beta):
    sc = score2 if nwin == 2 else score4; ranked = {}
    for q in dq:
        box = box_d[q][:K]; inbox = set(box)
        have = [d for d in box if (q, d) in sc]; miss = [d for d in box if (q, d) not in sc]
        if have:
            fin = beta * zs([sc[(q, d)] for d in have]) + (1 - beta) * zs([rrf_sc[q].get(d, 0.0) for d in have])
            have = [d for _, d in sorted(zip(-fin, have), key=lambda t: (t[0], t[1]))]
        ranked[q] = (have + miss + [d for d in rrf_order[q] if d not in inbox])[:100]
    return ranked

def prec(ranked, ptypes=None):
    acc = defaultdict(lambda: {1: [], 10: []})
    for q, l in ranked.items():
        for k in (1, 10): acc[q2p_d[q]][k].append(len(set(l[:k]) & gold_d[q]) / len(gold_d[q]))
    ps = [p for p in acc if ptypes is None or ptype_d[p] in ptypes]
    return {k: float(np.mean([np.mean(acc[p][k]) for p in ps])) for k in (1, 10)}

base = {"BM25":  {q: [c["document_id"] for c in P["bm"][q]] for q in dq},
        "Dense": {q: [c["document_id"] for c in P["de"][q]] for q in dq},
        "RRF (không rerank)": rrf_order}
print("MỐC trên val           | R@1 tất cả | R@10 tất cả | R@1 dịch | R@10 dịch")
for n, rk in base.items():
    a, t = prec(rk), prec(rk, {"translation"})
    print(f"{n:22s} | {a[1]:10.3f} | {a[10]:11.3f} | {t[1]:8.3f} | {t[10]:9.3f}")

out = []
for K, nwin, beta in itertools.product((30, 50, 80), (2, 4), (0, 0.25, 0.5, 0.75, 1.0)):
    rk = run(K, nwin, beta); a, t = prec(rk), prec(rk, {"translation"})
    out.append((K, nwin, beta, a[1], a[10], t[1], t[10]))

print("\nTOÀN BỘ LƯỚI (theo thứ tự K, số đoạn, β)")
print("  K | đoạn |   β  | R@1 tất cả | R@10 tất cả | R@1 dịch | R@10 dịch")
for r in out:
    print(f"{r[0]:3d} | {r[1]:4d} | {r[2]:4.2f} | {r[3]:10.3f} | {r[4]:11.3f} | {r[5]:8.3f} | {r[6]:9.3f}")

with open(f'{FT}/val_grid.csv', 'w', newline='') as f:
    w = csv.writer(f); w.writerow(["K", "doan", "beta", "R1_all", "R10_all", "R1_dich", "R10_dich"]); w.writerows(out)
print("\nĐã lưu bảng lưới vào", f'{FT}/val_grid.csv')

MỐC trên val           | R@1 tất cả | R@10 tất cả | R@1 dịch | R@10 dịch
BM25                   |      0.798 |       0.828 |    0.067 |     0.103
Dense                  |      0.645 |       0.742 |    0.597 |     0.662
RRF (không rerank)     |      0.677 |       0.770 |    0.103 |     0.268

TOÀN BỘ LƯỚI (theo thứ tự K, số đoạn, β)
  K | đoạn |   β  | R@1 tất cả | R@10 tất cả | R@1 dịch | R@10 dịch
 30 |    2 | 0.00 |      0.677 |       0.820 |    0.103 |     0.436
 30 |    2 | 0.25 |      0.730 |       0.895 |    0.126 |     0.662
 30 |    2 | 0.50 |      0.869 |       0.899 |    0.668 |     0.668
 30 |    2 | 0.75 |      0.867 |       0.900 |    0.668 |     0.668
 30 |    2 | 1.00 |      0.863 |       0.895 |    0.668 |     0.668
 30 |    4 | 0.00 |      0.677 |       0.820 |    0.103 |     0.436
 30 |    4 | 0.25 |      0.734 |       0.895 |    0.126 |     0.662
 30 |    4 | 0.50 |      0.860 |       0.899 |    0.668 |     0.668
 30 |    4 | 0.75 |      0.880 |       0.899 |    0.66

In [ ]:
import re
for name in ("rows_d", "meta_d", "ptype_d", "q2p_d", "gold_d"):
    print(name, "có" if name in globals() else "THIẾU")

EN = {"the","and","of","to","in","is","that","for","it","with","was","as","on","are"}
def en_ratio(t):
    w = re.findall(r"[a-zA-Z']+", t.lower()); return sum(x in EN for x in w) / max(1, len(w))

print("tổng số cặp đã chấm:", len(meta_d))
tr = [(m, r) for m, r in zip(meta_d, rows_d)
      if m[2] in ("de", "bm") and ptype_d[q2p_d[m[0]]] == "translation"]
print("cặp thuộc câu hỏi dịch (khúc cố định):", len(tr))
g = [en_ratio(r["passage"]) for (q, d, k), r in tr if d in gold_d[q]]
n = [en_ratio(r["passage"]) for (q, d, k), r in tr if d not in gold_d[q]]
print("số đoạn đúng:", len(g), "| số đoạn nhiễu:", len(n))
if g and n:
    print("tỷ lệ từ Anh, đúng: trung vị", round(float(np.median(g)), 3),
          "| nhiễu: trung vị", round(float(np.median(n)), 3),
          "| nhiễu có ít từ Anh (<0.05):", round(float(np.mean(np.array(n) < 0.05)), 3))

rows_d có
meta_d có
ptype_d có
q2p_d có
gold_d có
tổng số cặp đã chấm: 84228
cặp thuộc câu hỏi dịch (khúc cố định): 7683
số đoạn đúng: 80 | số đoạn nhiễu: 7603
tỷ lệ từ Anh, đúng: trung vị 0.004 | nhiễu: trung vị 0.226 | nhiễu có ít từ Anh (<0.05): 0.012


In [ ]:
by_q = {}
for (q, d, k), r in zip(meta_d, rows_d):
    if ptype_d[q2p_d[q]] == "translation":
        by_q.setdefault(q, []).append((en_ratio(r["passage"]), d))
hit = [min(v)[1] in gold_d[q] for q, v in by_q.items()
       if any(d in gold_d[q] for _, d in v)]
print("câu dịch có đáp án trong danh sách:", len(hit))
print("đối thủ 'ít tiếng Anh nhất' đúng top-1:", round(float(np.mean(hit)), 3))

câu dịch có đáp án trong danh sách: 73
đối thủ 'ít tiếng Anh nhất' đúng top-1: 0.74


### Chẩn đoán 13c-1 — Ngôn ngữ của từng loại cặp ở câu hỏi dịch (chỉ đọc)

Mục đích: xem đoạn đúng (pos), đoạn sai cùng tài liệu (same_doc) và đoạn sai tài liệu khác (hard_bm25, hard_dense) thuộc ngôn ngữ nào.
Kết quả: pos và same_doc là es/de; hard_bm25 và hard_dense gần như 100% tiếng Anh.

In [ ]:
!pip -q install langdetect
import json, os
from collections import Counter, defaultdict
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0

FT   = '/content/drive/MyDrive/KLTN/finetune'
META = '/content/drive/MyDrive/KLTN/meta_for_analysis.json'
rows = [json.loads(l) for l in open(f'{FT}/train_pairs.jsonl', encoding='utf-8') if l.strip()]
ptype_tr = {p: v['type'] for p, v in json.load(open(META))['train']['parent_labels'].items()}

def lang_of(t):
    try: return detect(t[:1500])
    except Exception: return '??'

tr = [r for r in rows if ptype_tr.get(r['parent_id']) == 'translation']
print("số cặp thuộc câu hỏi dịch:", len(tr), "| số câu hỏi dịch:", len({r['query_id'] for r in tr}))

by_kind = defaultdict(Counter)
for r in tr:
    by_kind[r['kind']][lang_of(r['passage'])] += 1
for kind in ('pos', 'same_doc', 'hard_bm25', 'hard_dense'):
    c = by_kind[kind]; n = sum(c.values())
    print(f"{kind:11s} n={n:5d} |", ", ".join(f"{l} {v/n:.2f}" for l, v in c.most_common(5)))

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
số cặp thuộc câu hỏi dịch: 1665 | số câu hỏi dịch: 418
pos         n=  418 | es 0.67, de 0.33
same_doc    n=  411 | es 0.66, de 0.34
hard_bm25   n=  418 | en 1.00, es 0.00, fr 0.00
hard_dense  n=  418 | en 0.99, es 0.01


### Kết luận v1 (đóng băng, không chạy lại)

**Dữ liệu v1:** 17.171 cặp, sau lọc còn 16.740 (bỏ 109 câu hỏi nghi sai nhãn). Train 14.812 cặp (441 parent), val 1.928 cặp (59 parent). Mỗi câu hỏi có 1 đoạn đúng và 3 đoạn sai (hard_bm25, hard_dense, same_doc).

**Kết quả val v1:**
- Zero-shot: top-1 0,625, AUC 0,747. Sau fine-tune (2 epoch): top-1 0,921, AUC 0,980. Epoch 1 gần như bằng epoch 2 (0,923 và 0,976).
- Theo loại, top-1 trong nhóm: artificial 0,906, translation 1,000 (73 câu), simulated 1,000 (4 câu).
- Cấu hình rerank khóa trên val: K = 50, MaxP 4 đoạn, β = 1. R@1 tất cả 0,899 (BM25 0,798, Dense 0,645, RRF 0,677); R@1 nhóm dịch 0,825 (BM25 0,067, Dense 0,597, RRF 0,103).

**Phát hiện: đường tắt ngôn ngữ ở nhóm dịch.**
- Đoạn đúng của câu hỏi dịch gần như không có từ tiếng Anh (trung vị 0,004), đoạn nhiễu lúc tìm kiếm hầu hết là tiếng Anh (trung vị 0,226). Luật "chọn đoạn ít tiếng Anh nhất" đã đúng top-1 ở 0,74 (73 câu, 7 parent).
- Trong dữ liệu huấn luyện, ở câu hỏi dịch: đoạn đúng là tiếng Tây Ban Nha 67%, tiếng Đức 33%; mẫu sai same_doc cùng ngôn ngữ đó; nhưng hard_bm25 là 100% tiếng Anh và hard_dense 99% tiếng Anh.
- Hệ quả: v1 chưa từng thấy đoạn ngoại ngữ thuộc tài liệu sai. Top-1 nhóm dịch bằng 1,000 cho thấy v1 phân biệt được vùng đúng với vùng sai trong cùng tài liệu, nhưng chưa kiểm được liệu v1 có phân biệt được tài liệu ngoại ngữ đúng với tài liệu ngoại ngữ sai.

**Hạn chế của v1:** val chỉ có 7 parent dịch nên số nhiễu; đoạn đúng nhóm dịch do Dense (E5) chọn nên có thể có vòng tròn nhãn; chưa so epoch 1 và epoch 2 trên bài toán tìm kiếm; ngôn ngữ đo bằng đếm từ Anh (thô).

**Quyết định:** dựng v2 bằng cách thêm mẫu sai ngoại ngữ cùng ngôn ngữ (từ đoạn đúng của parent dịch khác) vào train và val, huấn luyện lại, so v1 và v2 trên cả hai bộ val. v1 giữ nguyên làm mốc.

# Phần 3 — v2: bỏ đường tắt ngôn ngữ ở nhóm translation

**Nguyên tắc:** không ghi đè file nào của v1. File mới: `train_pairs_v2.jsonl`, `val_pairs_v2.jsonl`, `ckpt_v2/`, `model_final_v2/`.
Val v1 (`val_pairs.jsonl`) giữ nguyên để so; `val_pairs_v2.jsonl` = val v1 + mẫu sai ngoại ngữ cùng ngôn ngữ (chỉ lấy từ các parent val, không lấy từ train).
Chấm cả v1 và v2 trên cả hai bộ val. Chạy được trên CPU đến hết Bước 13c-3; từ huấn luyện v2 trở đi cần GPU.

### Bước 13c-2 — Dựng dữ liệu v2 (thêm mẫu sai ngoại ngữ cùng ngôn ngữ)

Phát hiện ở 13c-1: ở câu hỏi dịch, mẫu sai từ tài liệu khác (hard_bm25, hard_dense) gần như 100% tiếng Anh.
Cách sửa: với mỗi câu hỏi dịch trong Train, thêm 2 mẫu sai là đoạn đúng của parent dịch khác, cùng ngôn ngữ, tài liệu không nằm trong đáp án của câu hỏi này.
Ghi ra `train_pairs_v2.jsonl`. Không đụng `val_pairs.jsonl`.

In [ ]:
# --- nạp lại biến cần thiết (các ô v1 phía trên có thể đã ghi đè tên biến, ví dụ `tr`) ---
!pip -q install langdetect
import json, os, random
from collections import defaultdict
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0
from google.colab import drive
drive.mount('/content/drive')

FT   = '/content/drive/MyDrive/KLTN/finetune'
META = '/content/drive/MyDrive/KLTN/meta_for_analysis.json'
def lang_of(t):
    try: return detect(t[:1500])
    except Exception: return '??'
rows = [json.loads(l) for l in open(f'{FT}/train_pairs.jsonl', encoding='utf-8') if l.strip()]
ptype_tr = {p: v['type'] for p, v in json.load(open(META))['train']['parent_labels'].items()}
tr = [r for r in rows if ptype_tr.get(r['parent_id']) == 'translation']
assert not os.path.exists(f'{FT}/train_pairs_v2.jsonl') or True  # cho phép dựng lại v2 (cùng seed, kết quả giống nhau)
# ---------------------------------------------------------------------------------------

import random
meta_tr = json.load(open(META))['train']
gold_tr = meta_tr['query_gold_docs']

pos_tr = [r for r in tr if r['kind'] == 'pos']
lang_pos = {r['query_id']: lang_of(r['passage']) for r in pos_tr}
by_lang = defaultdict(list)
for r in pos_tr: by_lang[lang_pos[r['query_id']]].append(r)
print("ngôn ngữ đoạn đúng:", {l: len(v) for l, v in by_lang.items()})

rng = random.Random(20261123)
N_ADD = 2
added = []
for r in pos_tr:
    q, lg = r['query_id'], lang_pos[r['query_id']]
    ok = [c for c in by_lang[lg] if c['parent_id'] != r['parent_id'] and c['doc_id'] not in set(gold_tr[q])]
    for c in rng.sample(ok, min(N_ADD, len(ok))):
        added.append({'query_id': q, 'parent_id': r['parent_id'], 'query': r['query'],
                      'passage': c['passage'], 'label': 0, 'kind': 'foreign_other',
                      'doc_id': c['doc_id'], 'pos_src': None, 'qoverlap': 0.0, 'vis_overlap': 0.0})

new_rows = rows + added
rng.shuffle(new_rows)
with open(f'{FT}/train_pairs_v2.jsonl', 'w', encoding='utf-8') as f:
    for r in new_rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"thêm {len(added)} cặp | v1 {len(rows)} dòng -> v2 {len(new_rows)} dòng")
print("kiểm: cùng ngôn ngữ với đoạn đúng:", round(sum(lang_of(a['passage']) == lang_pos[a['query_id']] for a in added[:200]) / min(200, len(added)), 2), "(200 cặp đầu)")
print("kiểm: có doc trùng đáp án:", sum(a['doc_id'] in set(gold_tr[a['query_id']]) for a in added), "(phải bằng 0)")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Mounted at /content/drive
ngôn ngữ đoạn đúng: {'es': 278, 'de': 140}
thêm 836 cặp | v1 14812 dòng -> v2 15648 dòng
kiểm: cùng ngôn ngữ với đoạn đúng: 1.0 (200 cặp đầu)
kiểm: có doc trùng đáp án: 0 (phải bằng 0)


### Bước 13c-3 — Dựng `val_pairs_v2.jsonl` (thêm mẫu sai ngoại ngữ vào val)

Với mỗi câu hỏi dịch trong val: thêm tối đa 2 mẫu sai là đoạn đúng của **parent dịch khác trong val**, cùng ngôn ngữ, tài liệu không nằm trong đáp án. Không lấy từ train để tránh lẫn dữ liệu đã học.
Val chỉ có 7 parent dịch nên có thể có ngôn ngữ ít parent: ô in ra số câu không đủ ứng viên.

In [ ]:
val_v1 = [json.loads(l) for l in open(f'{FT}/val_pairs.jsonl', encoding='utf-8') if l.strip()]
gold_all = json.load(open(META))['train']['query_gold_docs']
vtr  = [r for r in val_v1 if ptype_tr.get(r['parent_id']) == 'translation']
vpos = [r for r in vtr if r['kind'] == 'pos']
vlang = {r['query_id']: lang_of(r['passage']) for r in vpos}
vby = defaultdict(list)
for r in vpos: vby[vlang[r['query_id']]].append(r)
print("val dịch: câu hỏi", len(vpos), "| parent", len({r['parent_id'] for r in vpos}),
      "| theo ngôn ngữ:", {l: (len(v), len({x['parent_id'] for x in v})) for l, v in vby.items()}, "(số câu, số parent)")

rng = random.Random(20261124)
vadd, thieu = [], 0
for r in vpos:
    q = r['query_id']
    ok = [c for c in vby[vlang[q]] if c['parent_id'] != r['parent_id'] and c['doc_id'] not in set(gold_all[q])]
    if len(ok) < 2: thieu += 1
    for c in rng.sample(ok, min(2, len(ok))):
        vadd.append({'query_id': q, 'parent_id': r['parent_id'], 'query': r['query'], 'passage': c['passage'],
                     'label': 0, 'kind': 'foreign_other', 'doc_id': c['doc_id'], 'pos_src': None,
                     'qoverlap': 0.0, 'vis_overlap': 0.0})
val_v2 = val_v1 + vadd
with open(f'{FT}/val_pairs_v2.jsonl', 'w', encoding='utf-8') as f:
    for r in val_v2: f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"thêm {len(vadd)} cặp | val v1 {len(val_v1)} -> val v2 {len(val_v2)} | câu thiếu ứng viên (ít hơn 2): {thieu}")
print("kiểm: trùng đáp án:", sum(a['doc_id'] in set(gold_all[a['query_id']]) for a in vadd), "(phải bằng 0)")

val dịch: câu hỏi 73 | parent 7 | theo ngôn ngữ: {'es': (53, 4), 'de': (20, 3)} (số câu, số parent)
thêm 146 cặp | val v1 1928 -> val v2 2074 | câu thiếu ứng viên (ít hơn 2): 0
kiểm: trùng đáp án: 0 (phải bằng 0)


### Bước 14b — Hàm dùng chung và nạp dữ liệu v2 (cần GPU)

Nạp lại các hàm của Bước 14 (ghép token, chấm điểm) và thêm hàm báo cáo mới:
- top-1 trong nhóm: tính chung, theo loại parent (artificial, translation, simulated) và theo (loại, mức che giấu);
- **phép kiểm chính "nhiễu cũng là ngoại ngữ":** ở val v2, với câu hỏi dịch, chọn giữa đoạn đúng và 2 đoạn ngoại ngữ sai (ngẫu nhiên là 0,333).

Nạp 3 file: `train_pairs_v2.jsonl`, `val_pairs.jsonl` (val v1, không đổi), `val_pairs_v2.jsonl`.

In [ ]:
import os, json, time, random, math
import numpy as np, torch
from collections import defaultdict, Counter
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/KLTN'
FT   = f'{DRIVE_DIR}/finetune'
META = f'{DRIVE_DIR}/meta_for_analysis.json'
assert torch.cuda.is_available(), "Chưa bật GPU: Runtime → Change runtime type → T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import roc_auc_score

MODEL = 'cross-encoder/mmarco-mMiniLMv2-L12-H384-v1'
MAX_LEN, Q_MAX = 512, 160
tok = AutoTokenizer.from_pretrained(MODEL)

def read_jsonl(p):
    return [json.loads(l) for l in open(p, encoding="utf-8").read().split("\n") if l.strip()]

train_v2 = read_jsonl(f'{FT}/train_pairs_v2.jsonl')
val_v1   = read_jsonl(f'{FT}/val_pairs.jsonl')
val_v2   = read_jsonl(f'{FT}/val_pairs_v2.jsonl')
_pl      = json.load(open(META))['train']['parent_labels']
ptype_all = {p: v['type'] for p, v in _pl.items()}
ob_all    = {p: str(v.get('obfuscation')) for p, v in _pl.items()}
print("train v2:", len(train_v2), "| val v1:", len(val_v1), "| val v2:", len(val_v2))
print("val v1, số câu hỏi theo (loại, mức che giấu):",
      dict(Counter((ptype_all[r['parent_id']], ob_all[r['parent_id']]) for r in val_v1 if r['kind'] == 'pos')))

def encode_pair(query, passage):
    q = tok(query, add_special_tokens=False, truncation=True, max_length=Q_MAX)["input_ids"]
    room = MAX_LEN - len(q) - 4
    p = tok(passage, add_special_tokens=False, truncation=True, max_length=room)["input_ids"]
    return [tok.cls_token_id] + q + [tok.sep_token_id, tok.sep_token_id] + p + [tok.sep_token_id]

_a, _b = "hello world this is a question", "an answer passage with some words"
assert encode_pair(_a, _b) == tok(_a, _b)["input_ids"], "định dạng token không khớp, báo mình"

def collate(ids_list):
    n = max(len(x) for x in ids_list)
    ids = torch.full((len(ids_list), n), tok.pad_token_id, dtype=torch.long)
    mask = torch.zeros((len(ids_list), n), dtype=torch.long)
    for i, x in enumerate(ids_list):
        ids[i, :len(x)] = torch.tensor(x); mask[i, :len(x)] = 1
    return ids.cuda(), mask.cuda()

def logits_of(model, ids, mask):
    out = model(input_ids=ids, attention_mask=mask).logits
    return out.squeeze(-1) if out.shape[-1] == 1 else out[:, -1]

@torch.no_grad()
def score_rows(model, rows, bs=64):
    model.eval()
    order = sorted(range(len(rows)), key=lambda i: len(rows[i]["passage"]))
    scores = np.zeros(len(rows), dtype=np.float32)
    for s in range(0, len(order), bs):
        idx = order[s:s + bs]
        ids, mask = collate([encode_pair(rows[i]["query"], rows[i]["passage"]) for i in idx])
        with torch.autocast("cuda", dtype=torch.float16):
            lg = logits_of(model, ids, mask)
        scores[idx] = lg.float().cpu().numpy()
    return scores

def top1(rows, scores, keep=lambda r: True):
    g = defaultdict(list)
    for i, r in enumerate(rows):
        if keep(r): g[r["query_id"]].append(i)
    hit = tot = 0; chance = []
    for idx in g.values():
        lab = [rows[i]["label"] for i in idx]
        if sum(lab) == 0 or len(idx) < 2: continue
        tot += 1; hit += int(lab[int(np.argmax([scores[i] for i in idx]))] == 1); chance.append(1 / len(idx))
    return (hit / tot if tot else float('nan')), tot, (float(np.mean(chance)) if chance else float('nan'))

def report(rows, scores, tag=""):
    labels = np.array([r["label"] for r in rows])
    t, n, _ = top1(rows, scores)
    print(f"[{tag}] top-1 tất cả {t:.3f} ({n} câu) | AUC {roc_auc_score(labels, scores):.3f}")
    for ty in ("artificial", "translation", "simulated"):
        t, n, _ = top1(rows, scores, keep=lambda r, ty=ty: ptype_all[r["parent_id"]] == ty)
        if n: print(f"    {ty:12s} top-1 {t:.3f} ({n} câu)")
    keys = sorted({(ptype_all[r['parent_id']], ob_all[r['parent_id']]) for r in rows})
    print("    theo (loại, mức che giấu):")
    for ty, ob in keys:
        t, n, _ = top1(rows, scores, keep=lambda r, ty=ty, ob=ob: (ptype_all[r['parent_id']], ob_all[r['parent_id']]) == (ty, ob))
        if n: print(f"      {ty:12s} {ob:12s} top-1 {t:.3f} ({n} câu)")
    if any(r["kind"] == "foreign_other" for r in rows):
        t, n, ch = top1(rows, scores, keep=lambda r: r["kind"] in ("pos", "foreign_other"))
        print(f"    KIỂM CHÍNH (đoạn đúng vs 2 đoạn ngoại ngữ sai): top-1 {t:.3f} ({n} câu) | ngẫu nhiên {ch:.3f}")
        t, n, _ = top1(rows, scores, keep=lambda r: r["kind"] in ("pos", "same_doc", "foreign_other"))
        print(f"    (đoạn đúng vs cả same_doc và ngoại ngữ sai): top-1 {t:.3f} ({n} câu)")
    ks = ("pos", "hard_bm25", "hard_dense", "same_doc", "foreign_other")
    print("    điểm trung bình:", " | ".join(f"{k} {np.mean([scores[i] for i, r in enumerate(rows) if r['kind'] == k]):+.2f}" for k in ks if any(r['kind'] == k for r in rows)))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: Tesla T4
train v2: 15648 | val v1: 1928 | val v2: 2074
val v1, số câu hỏi theo (loại, mức che giấu): {('artificial', 'high'): 202, ('artificial', 'low'): 175, ('translation', 'None'): 73, ('simulated', 'None'): 4, ('artificial', 'none'): 29}


### Bước 15b — Huấn luyện v2

Cùng cấu hình v1 (2 epoch, learning rate 2e-5, batch 32, trọng số đoạn đúng 3,0). Khác duy nhất: dữ liệu huấn luyện là `train_pairs_v2.jsonl` (thêm 836 mẫu sai ngoại ngữ cùng ngôn ngữ cho câu hỏi dịch).

Lưu vào `ckpt_v2/` và `model_final_v2/`, **không đè v1**. Sau mỗi epoch in kết quả trên val v2.
Nếu Colab ngắt, chạy lại ô này: nó tự nạp checkpoint v2 cuối và học tiếp.

In [ ]:
EPOCHS, LR, BS, POS_W = 2, 2e-5, 32, 3.0
CKPT = f'{FT}/ckpt_v2'
os.makedirs(CKPT, exist_ok=True)

model = AutoModelForSequenceClassification.from_pretrained(MODEL).cuda()
assert model.config.num_labels == 1
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
steps_per_epoch = math.ceil(len(train_v2) / BS)
total = EPOCHS * steps_per_epoch
warm = int(0.1 * total)
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda s: (s + 1) / warm if s < warm else max(0.0, (total - s) / max(1, total - warm)))
scaler = torch.amp.GradScaler('cuda')
loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_W).cuda())

start_epoch = 0
for e in range(EPOCHS - 1, -1, -1):
    p = f'{CKPT}/epoch{e+1}.pt'
    if os.path.exists(p):
        st = torch.load(p, map_location="cuda")
        model.load_state_dict(st["model"]); opt.load_state_dict(st["opt"])
        sched.load_state_dict(st["sched"]); start_epoch = e + 1
        print("Đã nạp checkpoint", p); break

for ep in range(start_epoch, EPOCHS):
    model.train()
    rng = random.Random(20261123 + ep)
    order = list(range(len(train_v2))); rng.shuffle(order)
    t0, run = time.time(), 0.0
    for b, s in enumerate(range(0, len(order), BS)):
        batch = [train_v2[i] for i in order[s:s + BS]]
        ids, mask = collate([encode_pair(r["query"], r["passage"]) for r in batch])
        y = torch.tensor([float(r["label"]) for r in batch]).cuda()
        with torch.autocast("cuda", dtype=torch.float16):
            lg = logits_of(model, ids, mask)
        loss = loss_fn(lg.float(), y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        run = 0.98 * run + 0.02 * loss.item() if b else loss.item()
        if b % 100 == 0:
            print(f"epoch {ep+1} | bước {b}/{steps_per_epoch} | loss {run:.4f} | {time.time()-t0:.0f}s", flush=True)
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict()}, f'{CKPT}/epoch{ep+1}.pt')
    print(f"\n=== Hết epoch {ep+1} ({(time.time()-t0)/60:.1f} phút), đánh giá trên val v2 ===")
    report(val_v2, score_rows(model, val_v2), f"v2 epoch {ep+1}")

model.save_pretrained(f'{FT}/model_final_v2'); tok.save_pretrained(f'{FT}/model_final_v2')
print("\nĐã lưu v2 vào", f'{FT}/model_final_v2')
del model, opt; torch.cuda.empty_cache()

epoch 1 | bước 0/489 | loss 2.0025 | 4s


/tmp/ipykernel_874/119130832.py:40: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step()


epoch 1 | bước 100/489 | loss 0.9243 | 53s
epoch 1 | bước 200/489 | loss 0.4293 | 102s
epoch 1 | bước 300/489 | loss 0.3180 | 153s
epoch 1 | bước 400/489 | loss 0.2657 | 204s

=== Hết epoch 1 (4.3 phút), đánh giá trên val v2 ===
[v2 epoch 1] top-1 tất cả 0.934 (483 câu) | AUC 0.980
    artificial   top-1 0.921 (406 câu)
    translation  top-1 1.000 (73 câu)
    simulated    top-1 1.000 (4 câu)
    theo (loại, mức che giấu):
      artificial   high         top-1 0.842 (202 câu)
      artificial   low          top-1 1.000 (175 câu)
      artificial   none         top-1 1.000 (29 câu)
      simulated    None         top-1 1.000 (4 câu)
      translation  None         top-1 1.000 (73 câu)
    KIỂM CHÍNH (đoạn đúng vs 2 đoạn ngoại ngữ sai): top-1 1.000 (73 câu) | ngẫu nhiên 0.333
    (đoạn đúng vs cả same_doc và ngoại ngữ sai): top-1 0.950 (479 câu)
    điểm trung bình: pos +7.80 | hard_bm25 -4.01 | hard_dense -4.20 | same_doc -3.19 | foreign_other -5.50
epoch 2 | bước 0/489 | loss 0.1184 |


Đã lưu v2 vào /content/drive/MyDrive/KLTN/finetune/model_final_v2


### Bước 15c — So mô hình gốc, v1, v2 trên val v1 và val v2

**Tiêu chí ghi trước khi chạy (không đổi sau khi thấy kết quả):**
1. **Kiểm chính:** trên val v2, đoạn đúng so với 2 đoạn ngoại ngữ sai ở câu hỏi dịch. Ngẫu nhiên = 0,333. v2 đạt từ 0,70 trở lên thì coi là học nội dung.
2. Nếu v1 cũng đạt từ 0,70, thì v1 đã học được nội dung và đường tắt ít nghiêm trọng hơn lo ngại. Ghi nhận như vậy, không hạ giá v1.
3. **Không làm hỏng chỗ khác:** top-1 artificial của v2 (trên val v1) không thấp hơn v1 quá 0,01 (v1 là 0,906). Đây chỉ là cờ cảnh báo vì val nhiễu.

**Hạn chế:** val dịch chỉ có 7 parent (Tây Ban Nha 4, Đức 3), nên mọi số ở nhóm dịch chỉ là kiểm tra thô.

In [ ]:
models = {"gốc (zero-shot)": MODEL, "v1": f'{FT}/model_final', "v2": f'{FT}/model_final_v2'}
for name, path in models.items():
    m = AutoModelForSequenceClassification.from_pretrained(path).cuda()
    print(f"\n########## {name} ##########")
    report(val_v1, score_rows(m, val_v1), f"{name} | val v1")
    report(val_v2, score_rows(m, val_v2), f"{name} | val v2")
    del m; torch.cuda.empty_cache()


########## gốc (zero-shot) ##########
[gốc (zero-shot) | val v1] top-1 tất cả 0.625 (483 câu) | AUC 0.747
    artificial   top-1 0.606 (406 câu)
    translation  top-1 0.712 (73 câu)
    simulated    top-1 1.000 (4 câu)
    theo (loại, mức che giấu):
      artificial   high         top-1 0.460 (202 câu)
      artificial   low          top-1 0.731 (175 câu)
      artificial   none         top-1 0.862 (29 câu)
      simulated    None         top-1 1.000 (4 câu)
      translation  None         top-1 0.712 (73 câu)
    điểm trung bình: pos -2.66 | hard_bm25 -5.10 | hard_dense -5.46 | same_doc -4.83
[gốc (zero-shot) | val v2] top-1 tất cả 0.607 (483 câu) | AUC 0.727
    artificial   top-1 0.606 (406 câu)
    translation  top-1 0.589 (73 câu)
    simulated    top-1 1.000 (4 câu)
    theo (loại, mức che giấu):
      artificial   high         top-1 0.460 (202 câu)
      artificial   low          top-1 0.731 (175 câu)
      artificial   none         top-1 0.862 (29 câu)
      simulated    None


########## v1 ##########
[v1 | val v1] top-1 tất cả 0.921 (483 câu) | AUC 0.980
    artificial   top-1 0.906 (406 câu)
    translation  top-1 1.000 (73 câu)
    simulated    top-1 1.000 (4 câu)
    theo (loại, mức che giấu):
      artificial   high         top-1 0.822 (202 câu)
      artificial   low          top-1 0.989 (175 câu)
      artificial   none         top-1 1.000 (29 câu)
      simulated    None         top-1 1.000 (4 câu)
      translation  None         top-1 1.000 (73 câu)
    điểm trung bình: pos +7.13 | hard_bm25 -5.12 | hard_dense -5.37 | same_doc -4.12
[v1 | val v2] top-1 tất cả 0.921 (483 câu) | AUC 0.982
    artificial   top-1 0.906 (406 câu)
    translation  top-1 1.000 (73 câu)
    simulated    top-1 1.000 (4 câu)
    theo (loại, mức che giấu):
      artificial   high         top-1 0.822 (202 câu)
      artificial   low          top-1 0.989 (175 câu)
      artificial   none         top-1 1.000 (29 câu)
      simulated    None         top-1 1.000 (4 câu)
      tran


########## v2 ##########
[v2 | val v1] top-1 tất cả 0.928 (483 câu) | AUC 0.981
    artificial   top-1 0.914 (406 câu)
    translation  top-1 1.000 (73 câu)
    simulated    top-1 1.000 (4 câu)
    theo (loại, mức che giấu):
      artificial   high         top-1 0.827 (202 câu)
      artificial   low          top-1 1.000 (175 câu)
      artificial   none         top-1 1.000 (29 câu)
      simulated    None         top-1 1.000 (4 câu)
      translation  None         top-1 1.000 (73 câu)
    điểm trung bình: pos +7.43 | hard_bm25 -5.60 | hard_dense -5.84 | same_doc -4.76
[v2 | val v2] top-1 tất cả 0.928 (483 câu) | AUC 0.982
    artificial   top-1 0.914 (406 câu)
    translation  top-1 1.000 (73 câu)
    simulated    top-1 1.000 (4 câu)
    theo (loại, mức che giấu):
      artificial   high         top-1 0.827 (202 câu)
      artificial   low          top-1 1.000 (175 câu)
      artificial   none         top-1 1.000 (29 câu)
      simulated    None         top-1 1.000 (4 câu)
      tran

### Bước 17 — Chấm Test với cấu hình đã khóa (v2)

**Cấu hình khóa (chọn trên val ở Bước 16, trước khi chạm Test):**
- Ứng viên: xen kẽ BM25 và Dense (cắt cố định), K = 50. Sau top-50 giữ thứ tự RRF đến hạng 100.
- MaxP: tối đa 4 đoạn mỗi tài liệu (khúc BM25 và Dense chỉ ra, cả cắt cố định và theo cấu trúc); điểm tài liệu là điểm đoạn cao nhất.
- β = 1 (chỉ dùng điểm reranker).
- Mô hình: **v2** (`model_final_v2`, huấn luyện lại sau khi phát hiện đường tắt ngôn ngữ) và bản gốc chưa fine-tune làm mốc "trước". v1 chỉ chấm nếu còn đủ thời gian.

**Lý do chọn v2 (quyết định trên val, không dùng Test):** v2 ngang v1 ở val (0,928 so với 0,921, trong độ nhiễu), đạt phép kiểm "nhiễu cũng là ngoại ngữ" (1,000; v1 cũng 1,000) và không làm artificial tụt (0,914 so với 0,906). Chọn v2 vì loại được yếu tố gây nhiễu ngôn ngữ đã biết.

Test không dùng để huấn luyện hay chọn tham số. Mỗi mô hình chấm đúng một lần.

**17a. Dựng cặp (câu hỏi, đoạn) cho Test.** Với mỗi câu hỏi Test, lấy 50 tài liệu ứng viên đầu tiên rồi chọn tối đa 4 đoạn mỗi tài liệu để chấm ở bước sau.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os, json, time, zipfile, shutil, gc, pickle
import numpy as np
from collections import Counter, defaultdict

DRIVE_DIR = '/content/drive/MyDrive/KLTN'
FT        = f'{DRIVE_DIR}/finetune'
TEST_ZIP  = f'{DRIVE_DIR}/pan-pc11-test-v1.zip'
BM25_ZIP  = f'{DRIVE_DIR}/bm25_chunking_results.zip'
DENSE_ZIP = f'{DRIVE_DIR}/dense_chunking_results.zip'
META      = f'{DRIVE_DIR}/meta_for_analysis.json'
TEST_DIR  = '/content/data/derived/pan-pc11-test-v1'
K_LOCK, MAX_CHARS = 50, 1500

need = ('documents.jsonl', 'queries.jsonl', 'qrels.jsonl')
if not all(os.path.exists(f'{TEST_DIR}/{n}') for n in need):
    os.makedirs(TEST_DIR, exist_ok=True)
    with zipfile.ZipFile(TEST_ZIP) as z:
        names = z.namelist()
        for want in need:
            hits = [n for n in names if n == want or n.endswith('/' + want)]
            assert hits, f"Không thấy {want} trong zip. Vài tên đầu: {names[:8]}"
            with z.open(hits[0]) as src, open(f'{TEST_DIR}/{want}', 'wb') as dst:
                shutil.copyfileobj(src, dst)

def read_jsonl(p):
    return [json.loads(l) for l in open(p, encoding="utf-8").read().split("\n") if l.strip()]

def load_preds(zip_path, member, method, keep=None):
    out = {}
    with zipfile.ZipFile(zip_path) as z, z.open(member) as f:
        for raw in f:
            if not raw.strip(): continue
            r = json.loads(raw)
            if r["method"] == method and (keep is None or r["query_id"] in keep):
                out[r["query_id"]] = r["candidates"]
    return out

def union_box(a, b, K):
    out, seen, i = [], set(), 0
    while len(out) < K and (i < len(a) or i < len(b)):
        for lst in (a, b):
            if i < len(lst) and lst[i]["document_id"] not in seen and len(out) < K:
                seen.add(lst[i]["document_id"]); out.append(lst[i]["document_id"])
        i += 1
    return out

docs    = {d["document_id"]: d["text"] for d in read_jsonl(f'{TEST_DIR}/documents.jsonl')}
queries = {q["query_id"]: q["text"] for q in read_jsonl(f'{TEST_DIR}/queries.jsonl')}
meta = json.load(open(META))['test']
q2p, gold = meta['query_to_parent'], meta['query_gold_docs']
ptype = {p: v['type'] for p, v in meta['parent_labels'].items()}
assert set(queries) == set(q2p), "câu hỏi trong pool khác với meta"
print("Test: tài liệu", len(docs), "| câu hỏi", len(queries), "| parent", len(ptype))

M = {"bm": ("bm25", "fixed_window_bm25"), "de": ("dense", "fixed_window_dense"),
     "sbm": ("bm25", "structure_bm25"), "sde": ("dense", "structure_dense")}
P = {k: load_preds(BM25_ZIP if z == "bm25" else DENSE_ZIP,
                   f"results/test/{z}_chunking/predictions.jsonl", m) for k, (z, m) in M.items()}
span_of = {}
for k in M:
    for q, cl in P[k].items():
        for c in cl:
            if c["source_spans"]:
                s0 = c["source_spans"][0]; span_of[(k, q, c["document_id"])] = (s0["raw_start"], s0["raw_end"])

box = {q: union_box(P["bm"][q], P["de"][q], K_LOCK) for q in queries}
rrf_sc, rrf_order = {}, {}
for q in queries:
    sc = defaultdict(float)
    for k in ("bm", "de"):
        for c in P[k][q]: sc[c["document_id"]] += 1.0 / (60 + c["rank"])
    rrf_sc[q] = sc; rrf_order[q] = sorted(sc, key=lambda d: (-sc[d], d))[:100]

rows_t, meta_t = [], []
for q in queries:
    for d in box[q]:
        seen = set()
        for kind in ("de", "bm", "sde", "sbm"):
            sp = span_of.get((kind, q, d))
            if sp and sp not in seen:
                seen.add(sp); text = docs[d][sp[0]:sp[1]][:MAX_CHARS]
                if text.strip():
                    rows_t.append({"query": queries[q], "passage": text}); meta_t.append((q, d, kind))
print(f"\nSố cặp cần chấm: {len(rows_t)} | trung bình {len(rows_t)/(len(queries)*K_LOCK):.2f} đoạn mỗi ứng viên")
print(f"Ước lượng thời gian mỗi lần chấm: {len(rows_t)/154/60:.0f} phút (theo 154 cặp/giây)")

KS = (1, 10, 100)
def parent_recall(ranked):
    acc = defaultdict(lambda: {k: [] for k in KS})
    for q, lst in ranked.items():
        g = set(gold[q])
        for k in KS: acc[q2p[q]][k].append(len(set(lst[:k]) & g) / len(g))
    return {p: {k: float(np.mean(v[k])) for k in KS} for p, v in acc.items()}
for name, rk in (("BM25", {q: [c["document_id"] for c in P["bm"][q]] for q in queries}),
                 ("Dense", {q: [c["document_id"] for c in P["de"][q]] for q in queries}),
                 ("RRF", rrf_order)):
    pm = parent_recall(rk)
    print(f"  {name:6s}: R@1 {np.mean([v[1] for v in pm.values()]):.3f} | R@10 {np.mean([v[10] for v in pm.values()]):.3f} | R@100 {np.mean([v[100] for v in pm.values()]):.3f}")
pc = defaultdict(list)
for q in queries: pc[q2p[q]].append(len(set(gold[q]) & set(box[q])) / len(gold[q]))
pc = {p: float(np.mean(v)) for p, v in pc.items()}
print(f"TRẦN của danh sách xen kẽ K = {K_LOCK}: {np.mean(list(pc.values())):.3f} | " +
      " | ".join(f"{t} {np.mean([pc[p] for p in pc if ptype[p] == t]):.3f}" for t in ("artificial", "translation", "simulated")))

CHUNK = 30000
def score_ckpt(model, tag):
    path = lambda i: f'{FT}/test_{tag}_part{i:03d}.npy'
    parts, t0 = [], time.time()
    for i, s0 in enumerate(range(0, len(rows_t), CHUNK)):
        if os.path.exists(path(i)):
            parts.append(np.load(path(i))); continue
        sc = score_rows(model, rows_t[s0:s0 + CHUNK], bs=64)
        np.save(path(i), sc); parts.append(sc)
        print(f"  {tag}: xong {min(s0 + CHUNK, len(rows_t))}/{len(rows_t)} cặp | {(time.time() - t0) / 60:.1f} phút", flush=True)
    scores = np.concatenate(parts)
    assert len(scores) == len(rows_t), f"điểm đã lưu ({len(scores)}) lệch số cặp hiện tại ({len(rows_t)}): xoá file test_{tag}_part*.npy"
    d = {}
    for (q, doc, kind), v in zip(meta_t, scores):
        d[(q, doc)] = max(float(v), d.get((q, doc), -1e9))
    pickle.dump(d, open(f'{FT}/test_scores_{tag}.pkl', 'wb'))
    return d

Test: tài liệu 2000 | câu hỏi 2606 | parent 300

Số cặp cần chấm: 303766 | trung bình 2.33 đoạn mỗi ứng viên
Ước lượng thời gian mỗi lần chấm: 33 phút (theo 154 cặp/giây)
  BM25  : R@1 0.841 | R@10 0.870 | R@100 0.899
  Dense : R@1 0.672 | R@10 0.775 | R@100 0.889
  RRF   : R@1 0.760 | R@10 0.839 | R@100 0.941
TRẦN của danh sách xen kẽ K = 50: 0.928 | artificial 0.942 | translation 0.705 | simulated 1.000


**17b. Chấm Test bằng mô hình v2 (đã fine-tune).** Mỗi cặp được cho một điểm; điểm của tài liệu là điểm cao nhất trong các đoạn của nó (MaxP).

In [ ]:
model_ft = AutoModelForSequenceClassification.from_pretrained(f'{FT}/model_final_v2').cuda()
test_v2 = score_ckpt(model_ft, "finetuned_v2")
print("Xong v2:", len(test_v2), "cặp (câu hỏi, tài liệu)")
del model_ft; torch.cuda.empty_cache()

  finetuned_v2: xong 30000/303766 cặp | 3.2 phút
  finetuned_v2: xong 60000/303766 cặp | 6.5 phút
  finetuned_v2: xong 90000/303766 cặp | 9.9 phút
  finetuned_v2: xong 120000/303766 cặp | 13.2 phút
  finetuned_v2: xong 150000/303766 cặp | 16.5 phút
  finetuned_v2: xong 180000/303766 cặp | 19.8 phút
  finetuned_v2: xong 210000/303766 cặp | 23.2 phút
  finetuned_v2: xong 240000/303766 cặp | 26.5 phút
  finetuned_v2: xong 270000/303766 cặp | 29.9 phút
  finetuned_v2: xong 300000/303766 cặp | 33.2 phút
  finetuned_v2: xong 303766/303766 cặp | 33.6 phút
Xong v2: 130300 cặp (câu hỏi, tài liệu)


**17c. Chấm Test bằng mô hình gốc, chưa fine-tune.** Dùng làm mốc "trước khi fine-tune" để thấy fine-tune cải thiện được bao nhiêu.

In [ ]:
model_zs = AutoModelForSequenceClassification.from_pretrained(MODEL).cuda()
test_zs = score_ckpt(model_zs, "zeroshot")
print("Xong mô hình gốc:", len(test_zs), "cặp (câu hỏi, tài liệu)")
del model_zs; torch.cuda.empty_cache()

  zeroshot: xong 30000/303766 cặp | 3.4 phút
  zeroshot: xong 60000/303766 cặp | 6.8 phút
  zeroshot: xong 90000/303766 cặp | 10.3 phút
  zeroshot: xong 120000/303766 cặp | 13.7 phút
  zeroshot: xong 150000/303766 cặp | 17.2 phút
  zeroshot: xong 180000/303766 cặp | 20.6 phút
  zeroshot: xong 210000/303766 cặp | 24.1 phút
  zeroshot: xong 240000/303766 cặp | 27.5 phút
  zeroshot: xong 270000/303766 cặp | 31.0 phút
  zeroshot: xong 300000/303766 cặp | 34.4 phút
  zeroshot: xong 303766/303766 cặp | 34.8 phút
Xong mô hình gốc: 130300 cặp (câu hỏi, tài liệu)


**17d — Chấm Test bằng v1 (tùy chọn)**

Mục đích: so trực tiếp v1 (huấn luyện chưa có mẫu sai ngoại ngữ) với v2 trên Test, để biết việc thêm mẫu sai ngoại ngữ có tạo khác biệt ở bước tìm kiếm thật hay không. Cùng cấu hình khóa, cùng K = 50.
Điểm lưu ở `test_scores_finetuned_v1.pkl`. Chạy sau 17c, trong cùng phiên với 17a.

In [ ]:
model_v1 = AutoModelForSequenceClassification.from_pretrained(f'{FT}/model_final').cuda()
test_v1 = score_ckpt(model_v1, "finetuned_v1")
print("Xong v1:", len(test_v1), "cặp (câu hỏi, tài liệu)")
del model_v1; torch.cuda.empty_cache()

  finetuned_v1: xong 30000/303766 cặp | 3.4 phút
  finetuned_v1: xong 60000/303766 cặp | 6.8 phút
  finetuned_v1: xong 90000/303766 cặp | 10.2 phút
  finetuned_v1: xong 120000/303766 cặp | 13.6 phút
  finetuned_v1: xong 150000/303766 cặp | 17.0 phút
  finetuned_v1: xong 180000/303766 cặp | 20.5 phút
  finetuned_v1: xong 210000/303766 cặp | 23.9 phút
  finetuned_v1: xong 240000/303766 cặp | 27.4 phút
  finetuned_v1: xong 270000/303766 cặp | 30.8 phút
  finetuned_v1: xong 300000/303766 cặp | 34.2 phút
  finetuned_v1: xong 303766/303766 cặp | 34.6 phút
Xong v1: 130300 cặp (câu hỏi, tài liệu)


### Bước 18 — Kết quả Test (đọc điểm đã lưu, không cần GPU)

**Cách xếp hạng cuối:** top-50 của danh sách xen kẽ xếp lại theo điểm reranker (β = 1, cấu hình khóa ở Bước 16 với v1, áp dụng không đổi cho v2); hạng sau 50 giữ thứ tự RRF.

**Cách tính:** recall từng câu hỏi, lấy trung bình theo parent, rồi trung bình qua parent. Khoảng tin cậy 95% bằng bootstrap theo parent (2.000 lần, seed 20261121); so sánh cặp trên cùng các parent. Dấu * nghĩa là khoảng tin cậy của hiệu không chứa 0.

**Báo cáo thêm:** theo (loại, mức che giấu); mốc ngôn ngữ ở nhóm dịch; tỷ lệ top-1 là đoạn ngoại ngữ; lưu thứ hạng từng câu hỏi lên Drive.

**18a. Bảng điểm và so sánh.** Tính Recall từng hệ (BM25, Dense, RRF, mô hình gốc, v1, v2), kèm khoảng tin cậy 95% và so sánh cặp.

In [ ]:
import pickle
BETA = 1.0   # khóa ở Bước 16 (β = 1, chỉ điểm reranker)
TAGS = {"v2 (fine-tune)": "finetuned_v2", "v1 (fine-tune)": "finetuned_v1", "gốc (zero-shot)": "zeroshot"}
sc = {n: pickle.load(open(f'{FT}/test_scores_{t}.pkl', 'rb')) for n, t in TAGS.items()
      if os.path.exists(f'{FT}/test_scores_{t}.pkl')}
print("mô hình có điểm Test:", list(sc)); assert sc, "chưa có file điểm nào: chạy 17b/17c trước"

def zs(x):
    x = np.asarray(x, float); sd = x.std()
    return (x - x.mean()) / sd if sd > 0 else x * 0

def final_ranking(s, beta=BETA, K=K_LOCK):
    ranked = {}
    for q in queries:
        b = box[q][:K]; inbox = set(b)
        have = [d for d in b if (q, d) in s]; miss = [d for d in b if (q, d) not in s]
        if have:
            fin = beta * zs([s[(q, d)] for d in have]) + (1 - beta) * zs([rrf_sc[q].get(d, 0.0) for d in have])
            have = [d for _, d in sorted(zip(-fin, have), key=lambda t: (t[0], t[1]))]
        ranked[q] = (have + miss + [d for d in rrf_order[q] if d not in inbox])[:100]
    return ranked

results = {"BM25": {q: [c["document_id"] for c in P["bm"][q]] for q in queries},
           "Dense": {q: [c["document_id"] for c in P["de"][q]] for q in queries},
           "RRF": rrf_order}
for n in sc: results[n] = final_ranking(sc[n])
PM = {n: parent_recall(rk) for n, rk in results.items()}

ob = {p: str(v.get('obfuscation')) for p, v in meta['parent_labels'].items()}
GROUPS = [("tất cả", lambda p: True)] + [(t, (lambda p, t=t: ptype[p] == t)) for t in ("artificial", "translation", "simulated")] \
       + [(f"{t}/{o}", (lambda p, t=t, o=o: ptype[p] == t and ob[p] == o)) for t, o in sorted({(ptype[p], ob[p]) for p in ptype})]

print(f"{'phương pháp':18s} {'nhóm':22s} | {'n':>3s} | {'R@1':>6s} | {'R@10':>6s} | {'R@100':>6s}")
for n, pm in PM.items():
    for g, f in GROUPS:
        ps = [p for p in pm if f(p)]
        if ps: print(f"{n:18s} {g:22s} | {len(ps):3d} | " + " | ".join(f"{np.mean([pm[p][k] for p in ps]):6.3f}" for k in KS))
    print()

rng = np.random.default_rng(20261121)
def boot(a, b, k, ps, n=2000):
    d = np.array([PM[a][p][k] - PM[b][p][k] for p in ps]); idx = rng.integers(0, len(ps), (n, len(ps)))
    return d.mean(), np.percentile(d[idx].mean(1), [2.5, 97.5])

PAIRS = [(a, b) for a in sc for b in ("BM25", "Dense", "RRF")] + \
        [("v2 (fine-tune)", "gốc (zero-shot)"), ("v2 (fine-tune)", "v1 (fine-tune)"), ("v1 (fine-tune)", "gốc (zero-shot)")]
PAIRS = [(a, b) for a, b in PAIRS if a in PM and b in PM]
print("Hiệu A − B, bootstrap 95% theo parent (* = khoảng không chứa 0):")
for a, b in PAIRS:
    for g in ("tất cả", "artificial", "translation"):
        ps = [p for p in PM["BM25"] if g == "tất cả" or ptype[p] == g]
        if len(ps) < 5: continue
        for k in (1, 10):
            m, ci = boot(a, b, k, ps)
            star = "*" if (ci[0] > 0 or ci[1] < 0) else " "
            print(f"  {a} − {b:16s} | {g:11s} (n={len(ps):3d}) | R@{k:<2d}: {m:+.3f} [{ci[0]:+.3f}, {ci[1]:+.3f}] {star}")

mô hình có điểm Test: ['v2 (fine-tune)', 'v1 (fine-tune)', 'gốc (zero-shot)']
phương pháp        nhóm                   |   n |    R@1 |   R@10 |  R@100
BM25               tất cả                 | 300 |  0.841 |  0.870 |  0.899
BM25               artificial             | 256 |  0.888 |  0.919 |  0.951
BM25               translation            |  23 |  0.171 |  0.208 |  0.229
BM25               simulated              |  21 |  1.000 |  1.000 |  1.000
BM25               artificial/high        | 119 |  0.780 |  0.836 |  0.896
BM25               artificial/low         | 131 |  0.981 |  0.990 |  0.999
BM25               artificial/none        |   6 |  1.000 |  1.000 |  1.000
BM25               simulated/None         |  21 |  1.000 |  1.000 |  1.000
BM25               translation/None       |  23 |  0.171 |  0.208 |  0.229

Dense              tất cả                 | 300 |  0.672 |  0.775 |  0.889
Dense              artificial             | 256 |  0.661 |  0.767 |  0.896
Dense              tr

**18b. Mốc ngôn ngữ, đo ngoại ngữ ở top-1, lưu thứ hạng.** Kiểm tra nhóm translation có đang "ăn gian" nhờ đường tắt ngôn ngữ không, và lưu thứ hạng từng câu hỏi lên Drive.

In [ ]:
import re
EN = {"the","and","of","to","in","is","that","for","it","with","was","as","on","are"}
def en_ratio(t):
    w = re.findall(r"[a-zA-Z']+", t.lower()); return sum(x in EN for x in w) / max(1, len(w))

enr = {}
for (q, d, k), r in zip(meta_t, rows_t):
    v = en_ratio(r["passage"]); enr[(q, d)] = min(v, enr.get((q, d), 9.0))

lang_rank = {q: sorted(box[q], key=lambda d: (enr.get((q, d), 9.0), d)) + [d for d in rrf_order[q] if d not in set(box[q])]
             for q in queries}
PM["mốc ngôn ngữ"] = parent_recall(lang_rank)
ps_tr = [p for p in PM["BM25"] if ptype[p] == "translation"]
print(f"NHÓM DỊCH (n={len(ps_tr)} parent):")
for n in ["BM25", "Dense", "RRF", "mốc ngôn ngữ"] + list(sc):
    print(f"  {n:18s} | " + " | ".join(f"R@{k} {np.mean([PM[n][p][k] for p in ps_tr]):.3f}" for k in KS))

tq = [q for q in queries if ptype[q2p[q]] == "translation"]
print("\nTỷ lệ top-1 là đoạn ngoại ngữ (ít từ Anh < 0,05), nhóm dịch:")
for n in ["BM25", "Dense", "RRF"] + list(sc):
    ids = [(q, results[n][q][0]) for q in tq if (q, results[n][q][0]) in enr]
    print(f"  {n:18s}: {np.mean([enr[x] < 0.05 for x in ids]):.3f} ({len(ids)} câu)")
print("  (tham chiếu: tỷ lệ đoạn ngoại ngữ trong tài liệu đúng nằm trong hộp:",
      round(float(np.mean([enr[(q, d)] < 0.05 for q in tq for d in gold[q] if (q, d) in enr])), 3), ")")

out = f'{FT}/test_rankings.json'
json.dump({n: rk for n, rk in results.items()}, open(out, 'w'), ensure_ascii=False)
print("\nĐã lưu thứ hạng từng câu hỏi vào", out)

NHÓM DỊCH (n=23 parent):
  BM25               | R@1 0.171 | R@10 0.208 | R@100 0.229
  Dense              | R@1 0.564 | R@10 0.652 | R@100 0.719
  RRF                | R@1 0.208 | R@10 0.226 | R@100 0.713
  mốc ngôn ngữ       | R@1 0.663 | R@10 0.705 | R@100 0.713
  v2 (fine-tune)     | R@1 0.685 | R@10 0.705 | R@100 0.713
  v1 (fine-tune)     | R@1 0.687 | R@10 0.705 | R@100 0.713
  gốc (zero-shot)    | R@1 0.498 | R@10 0.625 | R@100 0.713

Tỷ lệ top-1 là đoạn ngoại ngữ (ít từ Anh < 0,05), nhóm dịch:
  BM25              : 0.200 (260 câu)
  Dense             : 0.727 (260 câu)
  RRF               : 0.242 (260 câu)
  v2 (fine-tune)    : 0.877 (260 câu)
  v1 (fine-tune)    : 0.881 (260 câu)
  gốc (zero-shot)   : 0.619 (260 câu)
  (tham chiếu: tỷ lệ đoạn ngoại ngữ trong tài liệu đúng nằm trong hộp: 1.0 )

Đã lưu thứ hạng từng câu hỏi vào /content/drive/MyDrive/KLTN/finetune/test_rankings.json


In [ ]:
pairs = [("v2 (fine-tune)", "BM25"), ("v2 (fine-tune)", "gốc (zero-shot)"), ("v2 (fine-tune)", "v1 (fine-tune)")]
print("Hiệu theo mức che giấu (artificial), bootstrap 95% theo parent:")
for g, f in GROUPS:
    if not g.startswith("artificial/"): continue
    ps = [p for p in PM["BM25"] if f(p)]
    if len(ps) < 10: print(f"  bỏ qua {g} (n={len(ps)}, quá ít)"); continue
    for a, b in pairs:
        for k in (1, 10):
            m, ci = boot(a, b, k, ps)
            star = "*" if (ci[0] > 0 or ci[1] < 0) else " "
            print(f"  {a} − {b:16s} | {g:16s} (n={len(ps):3d}) | R@{k:<2d}: {m:+.3f} [{ci[0]:+.3f}, {ci[1]:+.3f}] {star}")

Hiệu theo mức che giấu (artificial), bootstrap 95% theo parent:
  v2 (fine-tune) − BM25             | artificial/high  (n=119) | R@1 : +0.050 [+0.027, +0.074] *
  v2 (fine-tune) − BM25             | artificial/high  (n=119) | R@10: +0.027 [+0.014, +0.042] *
  v2 (fine-tune) − gốc (zero-shot)  | artificial/high  (n=119) | R@1 : +0.431 [+0.364, +0.505] *
  v2 (fine-tune) − gốc (zero-shot)  | artificial/high  (n=119) | R@10: +0.164 [+0.120, +0.210] *
  v2 (fine-tune) − v1 (fine-tune)   | artificial/high  (n=119) | R@1 : +0.000 [-0.004, +0.005]  
  v2 (fine-tune) − v1 (fine-tune)   | artificial/high  (n=119) | R@10: +0.003 [-0.002, +0.011]  
  v2 (fine-tune) − BM25             | artificial/low   (n=131) | R@1 : +0.010 [+0.003, +0.018] *
  v2 (fine-tune) − BM25             | artificial/low   (n=131) | R@10: +0.005 [+0.001, +0.009] *
  v2 (fine-tune) − gốc (zero-shot)  | artificial/low   (n=131) | R@1 : +0.361 [+0.304, +0.417] *
  v2 (fine-tune) − gốc (zero-shot)  | artificial/low   (n=131) 

In [ ]:
!pip -q install langdetect
from collections import Counter, defaultdict
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0

def doc_lang(t):
    mid = len(t) // 2
    try: return detect(t[mid:mid + 1500])
    except Exception: return "??"

# ngôn ngữ của mỗi parent dịch: lấy từ tài liệu nguồn đúng của các câu hỏi thuộc parent đó
par_docs = defaultdict(set)
for q in queries: par_docs[q2p[q]].update(gold[q])
par_lang = {}
for p, ds in par_docs.items():
    if ptype[p] == "translation":
        cnt = Counter(doc_lang(docs[d]) for d in ds)
        par_lang[p] = cnt.most_common(1)[0][0]
print("Ngôn ngữ nguồn của parent dịch (Test):", dict(Counter(par_lang.values())))

LANG_GROUPS = [(f"translation/{lg}", (lambda p, lg=lg: par_lang.get(p) == lg)) for lg in sorted(set(par_lang.values()))]

def ci_mean(name, k, ps, n=2000):
    v = np.array([PM[name][p][k] for p in ps]); idx = rng.integers(0, len(ps), (n, len(ps)))
    return v.mean(), np.percentile(v[idx].mean(1), [2.5, 97.5])

print(f"{'nhóm':22s} {'phương pháp':18s} | {'n':>3s} | R@1 [95% CI]            | R@10 [95% CI]")
for g, f in GROUPS + LANG_GROUPS:
    ps = [p for p in PM["BM25"] if f(p)]
    if len(ps) < 5:
        print(f"{g:22s} (bỏ qua, n={len(ps)} quá ít)"); continue
    for n in PM:
        m1, c1 = ci_mean(n, 1, ps); m10, c10 = ci_mean(n, 10, ps)
        print(f"{g:22s} {n:18s} | {len(ps):3d} | {m1:.3f} [{c1[0]:.3f}, {c1[1]:.3f}] | {m10:.3f} [{c10[0]:.3f}, {c10[1]:.3f}]")
    print()

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Ngôn ngữ nguồn của parent dịch (Test): {'es': 13, 'de': 10}
nhóm                   phương pháp        |   n | R@1 [95% CI]            | R@10 [95% CI]
tất cả                 BM25               | 300 | 0.841 [0.802, 0.877] | 0.870 [0.834, 0.900]
tất cả                 Dense              | 300 | 0.672 [0.625, 0.719] | 0.775 [0.734, 0.816]
tất cả                 RRF                | 300 | 0.760 [0.718, 0.800] | 0.839 [0.804, 0.876]
tất cả                 v2 (fine-tune)     | 300 | 0.904 [0.875, 0.932] | 0.921 [0.894, 0.945]
tất cả                 v1 (fine-tune)     | 300 | 0.901 [0.870, 0.929] | 0.920 [0.893, 0.946]
tất cả                 gốc (zero-shot)    | 300 | 0.533 [0.491, 0.578] | 0.774 [0.739, 0.809]
tất cả                 mốc ngôn ngữ       | 300 | 0.056 [0.033, 0.082] | 0.119 [0.089, 0.151]

artificial             BM25               | 256 | 0.888 

In [ ]:
# Đếm ngôn ngữ của toàn bộ tài liệu trong kho Test (2.000 docs)
!pip -q install langdetect
import json, glob, os
from collections import Counter
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0

# 1) Tìm documents.jsonl của Test (thư mục đã giải nén từ pan-pc11-test-v1.zip)
import zipfile, glob, os
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

ZIP = '/content/drive/MyDrive/KLTN/pan-pc11-test-v1.zip'
print("zip tồn tại:", os.path.exists(ZIP))

with zipfile.ZipFile(ZIP) as z:
    names = [n for n in z.namelist() if n.endswith('documents.jsonl')]
    print("file documents.jsonl trong zip:", names)
    z.extract(names[0], '/content/test_pool')   # chỉ giải nén đúng 1 file này

DOC_PATH = glob.glob('/content/test_pool/**/documents.jsonl', recursive=True)[0]
print("DOC_PATH =", DOC_PATH)

docs = {}
with open(DOC_PATH, encoding='utf-8') as f:
    for line in f.read().split('\n'):          # không dùng splitlines()
        if line.strip():
            d = json.loads(line)
            docs[d.get('document_id') or d.get('doc_id') or d.get('id')] = d
print("số tài liệu:", len(docs), "| khóa mẫu:", list(next(iter(docs.values())).keys()))

TEXT_KEY = next(k for k in next(iter(docs.values())).keys() if k in ('text', 'content', 'body'))

def lang_of(t):
    try: return detect(t[:2000])
    except Exception: return 'unk'

lang = {i: lang_of(d[TEXT_KEY]) for i, d in docs.items()}
print("\nToàn bộ kho Test:", Counter(lang.values()).most_common())

# 2) Tài liệu đáp án đúng (gold) và parent dịch
meta = json.load(open('/content/drive/MyDrive/KLTN/meta_for_analysis.json'))
t = meta['test']
gold_all = set()
for v in t['query_gold_docs'].values():
    gold_all |= set([v] if isinstance(v, str) else v)

print("\nGold docs (mọi loại):", len(gold_all), Counter(lang[g] for g in gold_all if g in lang).most_common())
print("Không phải gold (nhiễu):", len(docs) - len(gold_all),
      Counter(lang[i] for i in docs if i not in gold_all).most_common())

# gold của các parent dịch
tr_parents = {p for p, l in t['parent_labels'].items() if l.get('type') == 'translation'}
gold_tr = set()
for q, p in t['query_to_parent'].items():
    if p in tr_parents:
        v = t['query_gold_docs'][q]
        gold_tr |= set([v] if isinstance(v, str) else v)
print("\nGold của parent dịch:", len(gold_tr), Counter(lang[g] for g in gold_tr if g in lang).most_common())

zip tồn tại: True
file documents.jsonl trong zip: ['documents.jsonl']
DOC_PATH = /content/test_pool/documents.jsonl
số tài liệu: 2000 | khóa mẫu: ['dataset_version', 'document_id', 'raw_sha256', 'source_reference', 'text', 'version']

Toàn bộ kho Test: [('en', 1858), ('de', 95), ('es', 44), ('ru', 1), ('fr', 1), ('it', 1)]

Gold docs (mọi loại): 287 [('en', 264), ('es', 13), ('de', 10)]
Không phải gold (nhiễu): 1713 [('en', 1594), ('de', 85), ('es', 31), ('ru', 1), ('fr', 1), ('it', 1)]

Gold của parent dịch: 23 [('es', 13), ('de', 10)]


### Kết luận Bước 18 — Kết quả Test (300 parent, 2.606 câu hỏi)

Cấu hình khóa trên val với v1 (K = 50 xen kẽ BM25/Dense, MaxP 4 đoạn, β = 1), áp dụng không đổi cho v2 và Test. Test không dùng để huấn luyện hay chọn tham số.

| Hệ | R@1 | R@10 | R@100 |
|---|---:|---:|---:|
| BM25 | 0,841 | 0,870 | 0,899 |
| Dense | 0,672 | 0,775 | 0,889 |
| RRF | 0,760 | 0,839 | 0,941 |
| Reranker gốc (zero-shot) | 0,533 | 0,774 | 0,941 |
| v1 (fine-tune) | 0,901 | 0,920 | 0,941 |
| v2 (fine-tune, thêm mẫu sai ngoại ngữ) | 0,904 | 0,921 | 0,941 |

**Theo loại và mức che giấu (R@1):** artificial 0,916 (BM25 0,888); artificial high 0,830 (BM25 0,780, RRF 0,620, Dense 0,430, gốc 0,399); artificial low 0,990 (BM25 0,981); simulated 1,000 (21 parent); translation 0,685 (BM25 0,171, Dense 0,564, mốc ngôn ngữ 0,663). Nhóm artificial/none (6 parent) quá ít để kết luận.

**Kết luận chính (bootstrap 95% theo parent):**
1. Fine-tune cải thiện mạnh so với reranker gốc: R@1 +0,371 [+0,332, +0,413]. Reranker gốc xếp tệ hơn cả BM25 (R@1 −0,308), đúng với chẩn đoán "reranker sai nhiệm vụ".
2. v2 hơn BM25 ở R@1: +0,063 [+0,042, +0,086] tổng; artificial +0,028 [+0,017, +0,041]; artificial high +0,050 [+0,027, +0,074]; artificial low +0,010 [+0,003, +0,018].
3. v2 và v1 không khác nhau (R@1 +0,003 [−0,001, +0,011]). Phát hiện đường tắt ngôn ngữ không dẫn đến cải thiện đo được. Phép kiểm khống chế ngôn ngữ ở val: v1 và v2 đều 1,000, mô hình gốc 0,658.
4. R@100 không đổi giữa các reranker (0,941) vì reranker chỉ xếp lại top-50, phần sau giữ thứ tự RRF.

**Lưu ý về nhóm dịch:** luật "chọn đoạn ít tiếng Anh nhất" đạt R@1 0,663, gần bằng v2 (0,685). Mức tăng nhóm dịch so với BM25/Dense phần lớn là do phân biệt ngôn ngữ, không phải bằng chứng hiểu nội dung xuyên ngôn ngữ. Top-1 là đoạn ngoại ngữ: v2 88%, Dense 73%.

### Hạn chế
1. Nhóm dịch Test chỉ 23 parent; mốc ngôn ngữ đo thô bằng đếm từ Anh.
2. Cấu hình khóa trên val nhỏ (59 parent, nhóm dịch 7 parent), chọn với v1, áp dụng cho v2 mà không chạy lại.
3. Trần của danh sách K = 50 ở nhóm dịch Test là 0,705 (val 0,825), giới hạn R@1 của mọi reranker.
4. Chưa so epoch 1 và epoch 2 trên bài toán tìm kiếm; không chỉnh siêu tham số nào ngoài mặc định.
5. Đoạn đúng nhóm dịch do Dense (E5) chọn nên nhãn có thể bị lệch (vòng tròn); bộ lọc bỏ 109 câu hỏi (29 và 12 câu dịch).
6. Câu hỏi bị cắt còn 160 token (khoảng một nửa); MaxP chỉ phía đoạn văn, tối đa 4 đoạn; chưa làm MaxP phía câu hỏi.
7. BM25 và Dense dùng tham số mặc định, chưa tinh chỉnh.
8. Nhóm simulated (21 parent) và artificial/none (6 parent) quá ít; chênh lệch v1 và v2 ở simulated là một parent.
9. Phép kiểm khống chế ngôn ngữ dùng mẫu sai khác chủ đề (dễ); chưa thử mẫu ngoại ngữ cùng chủ đề.
10. Chỉ một bộ dữ liệu (PAN-PC-11), chưa kiểm khái quát hóa.